In [ ]:
# Cell 1 - imports
import json
import os
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from IPython.display import Image as DisplayImage, Video, display
from PIL import Image, ImageDraw


In [ ]:
# Display and detection helpers
VIDEO_DIR = PROJECT_ROOT / "generated_videos"
VIDEO_DIR.mkdir(exist_ok=True)
def save_video(frames, path, fps=20):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = frames.astype(np.uint8)
    try:
        imageio.mimsave(path, frames, fps=fps)
        return path
    except Exception as exc:
        fallback = path.with_suffix(".gif")
        print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
        print(f"Saving GIF fallback: {fallback}")
        imageio.mimsave(fallback, frames, duration=1 / fps)
        return fallback
def show_video(frames, name, fps=20, embed=True):
    path = save_video(frames, VIDEO_DIR / name, fps=fps)
    if path.suffix.lower() == ".gif":
        display(DisplayImage(filename=str(path)))
    else:
        display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def show_video_file(path, embed=True):
    path = Path(path)
    display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def clip_indices(num_frames, clip_frames=48, seed=0):
    if num_frames <= 0:
        return np.asarray([], dtype=int)
    clip_frames = min(int(clip_frames), int(num_frames))
    rng = np.random.default_rng(seed)
    max_start = max(0, int(num_frames) - clip_frames)
    start = int(rng.integers(0, max_start + 1)) if max_start else 0
    return np.arange(start, start + clip_frames, dtype=int)
def bbox_from_mask(mask, min_area=50):
    mask = mask.astype(bool)
    visited = np.zeros(mask.shape, dtype=bool)
    boxes = []
    height, width = mask.shape
    ys, xs = np.nonzero(mask)
    for y0, x0 in zip(ys, xs):
        if visited[y0, x0] or not mask[y0, x0]:
            continue
        stack = [(int(y0), int(x0))]
        visited[y0, x0] = True
        x_min = x_max = int(x0)
        y_min = y_max = int(y0)
        area = 0
        while stack:
            y, x = stack.pop()
            area += 1
            x_min = min(x_min, x)
            x_max = max(x_max, x)
            y_min = min(y_min, y)
            y_max = max(y_max, y)
            for ny in (y - 1, y, y + 1):
                for nx in (x - 1, x, x + 1):
                    if ny == y and nx == x:
                        continue
                    if 0 <= ny < height and 0 <= nx < width and mask[ny, nx] and not visited[ny, nx]:
                        visited[ny, nx] = True
                        stack.append((ny, nx))
        if area >= min_area:
            boxes.append((x_min, y_min, x_max + 1, y_max + 1, float(area)))
    return boxes
def draw_boxes(ax, boxes, color="lime", labels=None):
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2, edgecolor=color)
        ax.add_patch(rect)
        if labels:
            ax.text(x1, max(0, y1 - 4), labels[i], color=color, fontsize=9, weight="bold")
def show_detection_frame(frames, frame_idx, boxes, title, labels=None):
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(frames[frame_idx])
    draw_boxes(ax, boxes, labels=labels)
    ax.set_title(title)
    ax.axis("off")
    plt.show()
def plot_detection_metric(values, title, ylabel="objects"):
    plt.figure(figsize=(8, 3))
    plt.plot(values)
    plt.title(title)
    plt.xlabel("frame")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:
# Internet video helpers for independent visual problems 4 and 5
def download_video(url, path):
    import urllib.request
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)
    return path
def read_video(path, max_frames=None, stride=1, start_frame=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    if start_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
    frames_out = []
    frame_no = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_no % stride == 0:
            frames_out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames_out) >= max_frames:
                break
        frame_no += 1
    cap.release()
    if not frames_out:
        raise RuntimeError(f"No frames read from {path}")
    return np.stack(frames_out)
def read_video_sample(path, max_frames=120, stride=3, start_seconds=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    return read_video(path, max_frames=max_frames, stride=stride, start_frame=int(start_seconds * fps))


# Visual Problems


## 1 - Lunar Lander


In [ ]:
# LunarLander setup - use kernel: Python (Action Inference)
import sys
import gymnasium as gym
from gymnasium.envs.box2d.lunar_lander import heuristic
ACTION_NAMES = {
    0: "noop",
    1: "left_engine",
    2: "main_engine",
    3: "right_engine",
}
print(sys.executable)
print(gym.__version__)


In [ ]:
# LunarLander rollout
def rollout_lunar_lander(policy, seed=0, max_steps=300, render_mode="rgb_array"):
    env = gym.make("LunarLander-v3", render_mode=render_mode)
    obs, info = env.reset(seed=seed)
    observations, actions, rewards, frames, infos = [], [], [], [], []
    for t in range(max_steps):
        action = int(policy(env.unwrapped, obs))
        next_obs, reward, terminated, truncated, info = env.step(action)
        observations.append(obs.copy())
        actions.append(action)
        rewards.append(float(reward))
        infos.append(info)
        frames.append(env.render())
        obs = next_obs
        if terminated or truncated:
            break
    env.close()
    return {
        "observations": np.asarray(observations, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.int64),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "frames": np.stack(frames),
        "infos": infos,
    }
rollout = rollout_lunar_lander(heuristic, seed=0)
frames = rollout["frames"]
actions = rollout["actions"]
observations = rollout["observations"]
print("frames:", frames.shape)
print("observations:", observations.shape)
print("actions:", actions.shape)
print("action counts:", {ACTION_NAMES[i]: int((actions == i).sum()) for i in ACTION_NAMES})


In [ ]:
frame_idx = min(10, len(frames) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames[frame_idx])
plt.title(f"t={frame_idx}, action={actions[frame_idx]} ({ACTION_NAMES[int(actions[frame_idx])]})")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames, "1_lunar_lander.mp4", fps=30)


## 2 - Car Racing


In [ ]:
# Install once if Box2D is missing
!pip install gymnasium[box2d] -q


In [ ]:
# CarRacing with random driving
import gymnasium as gym
env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
obs, _ = env.reset(seed=0)
frames_car = []
for t in range(300):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    frames_car.append(env.render())
    if terminated or truncated:
        break
env.close()
frames_car = np.stack(frames_car)
frames_car.shape


In [ ]:
car_frame_idx = min(40, len(frames_car) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames_car[car_frame_idx])
plt.title(f"CarRacing frame {car_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_car, "2_car_racing.mp4", fps=30)


## 3 - Recorded Traffic With People


In [ ]:
# Load recorded traffic video
import cv2
import urllib.request
if not PROJECT_ROOT / "external_videos/traffic.avi").exists():
    url = "https://raw.githubusercontent.com/opencv/opencv_extra/master/testdata/cv/video/768x576.avi"
    urllib.request.urlretrieve(url, "external_videos/traffic.avi")
cap = cv2.VideoCapture("external_videos/traffic.avi")
frames_medium = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames_medium.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()
frames_medium = np.stack(frames_medium)
frames_medium.shape


In [ ]:
traffic_frame_idx = min(1, len(frames_medium) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_medium[traffic_frame_idx])
plt.title(f"Recorded traffic frame {traffic_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_medium, "3_recorded_traffic_people.mp4", fps=20)


## 4 - Random YouTube Driving Scene


In [ ]:
# Problem 4: random 3-minute YouTube driving clip
PROBLEM_4_URL = "https://www.youtube.com/watch?v=7EovwWQIvBo"
problem_4_path = PROJECT_ROOT / "external_videos/problem_4_youtube_random.mp4")
if not problem_4_path.exists():
    raise FileNotFoundError(f"Missing {problem_4_path}. Download it with yt-dlp first.")
# Display the full downloaded 3-minute clip.
show_video_file(problem_4_path, embed=False)
# Use a small cached sample for plotting/models. Do not decode the full 3-minute file every run.
if "frames_problem4" not in globals():
    frames_problem4 = read_video_sample(problem_4_path, max_frames=120, stride=3, start_seconds=20)
problem4_frame_idx = min(30, len(frames_problem4) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem4[problem4_frame_idx])
plt.title(f"Problem 4 sampled frame {problem4_frame_idx}")
plt.axis("off")
plt.show()


## 5 - Hard Vehicle-Crowd Interaction


In [ ]:
# Problem 5: hardest 3-minute vehicle-crowd YouTube clip
PROBLEM_5_URL = "https://www.youtube.com/watch?v=7HaJArMDKgI"
problem_5_path = PROJECT_ROOT / "external_videos/problem_5_youtube_hardest.mp4")
if not problem_5_path.exists():
    raise FileNotFoundError(f"Missing {problem_5_path}. Download it with yt-dlp first.")
# Display the full downloaded 3-minute clip.
show_video_file(problem_5_path, embed=False)
# Use a small cached sample for plotting/models. Do not decode the full 3-minute file every run.
if "frames_problem5" not in globals():
    frames_problem5 = read_video_sample(problem_5_path, max_frames=120, stride=3, start_seconds=20)
problem5_frame_idx = min(40, len(frames_problem5) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem5[problem5_frame_idx])
plt.title(f"Problem 5 sampled frame {problem5_frame_idx}")
plt.axis("off")
plt.show()


# Object Detection


In [ ]:
# YOLO setup and display utilities. Run this once before the detection subsections.
!pip install ultralytics -q
from ultralytics import YOLO
import pandas as pd
model = YOLO("models/yolo11n.pt")
def run_yolo(frames_batch):
    results = model(list(frames_batch), stream=True, verbose=False)
    all_detections = []
    for r in results:
        boxes = r.boxes
        frame_dets = [
            (int(c), float(conf), *map(float, xyxy))
            for c, conf, xyxy in zip(boxes.cls, boxes.conf, boxes.xyxy)
        ]
        all_detections.append(frame_dets)
    return all_detections
def yolo_dets(frame_dets, conf_min=0.25):
    boxes, labels = [], []
    for class_id, conf, x1, y1, x2, y2 in frame_dets:
        if conf >= conf_min:
            boxes.append((x1, y1, x2, y2, conf))
            labels.append(f"{model.names[int(class_id)]} {conf:.2f}")
    return boxes, labels
def yolo_detection_counts(all_detections, conf_min=0.25):
    return np.asarray([sum(1 for _, conf, *_ in frame_dets if conf >= conf_min) for frame_dets in all_detections], dtype=int)
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter
    return float(inter / denom) if denom > 0 else 0.0
def track_yolo_detections(detections_batch, conf_min=0.25, iou_threshold=0.3, max_missing=2):
    tracked_frames = []
    active_tracks = []
    next_track_id = 0
    for frame_no, frame_dets in enumerate(detections_batch):
        old_track_count = len(active_tracks)
        detections = []
        for class_id, conf, x1, y1, x2, y2 in frame_dets:
            if conf < conf_min:
                continue
            detections.append({
                "frame": frame_no,
                "class_id": int(class_id),
                "class_name": model.names[int(class_id)],
                "confidence": float(conf),
                "box": (float(x1), float(y1), float(x2), float(y2)),
            })
        matches = []
        used_tracks = set()
        used_dets = set()
        candidates = []
        for track_idx, track in enumerate(active_tracks):
            if track["missing"] > max_missing:
                continue
            for det_idx, det in enumerate(detections):
                if det["class_id"] != track["class_id"]:
                    continue
                iou_score = box_iou(track["box"], det["box"])
                if iou_score >= iou_threshold:
                    candidates.append((iou_score, track_idx, det_idx))
        for _, track_idx, det_idx in sorted(candidates, reverse=True):
            if track_idx in used_tracks or det_idx in used_dets:
                continue
            matches.append((track_idx, det_idx))
            used_tracks.add(track_idx)
            used_dets.add(det_idx)
        frame_records = []
        for track_idx, det_idx in matches:
            track = active_tracks[track_idx]
            det = detections[det_idx]
            det["track_id"] = track["track_id"]
            det["matched_iou"] = box_iou(track["box"], det["box"])
            track.update({"box": det["box"], "class_id": det["class_id"], "missing": 0})
            frame_records.append(det)
        for det_idx, det in enumerate(detections):
            if det_idx in used_dets:
                continue
            det["track_id"] = next_track_id
            det["matched_iou"] = np.nan
            active_tracks.append({"track_id": next_track_id, "class_id": det["class_id"], "box": det["box"], "missing": 0})
            next_track_id += 1
            frame_records.append(det)
        for track_idx, track in enumerate(active_tracks[:old_track_count]):
            if track_idx not in used_tracks:
                track["missing"] += 1
        active_tracks = [track for track in active_tracks if track["missing"] <= max_missing]
        tracked_frames.append(sorted(frame_records, key=lambda obj: obj["track_id"]))
    return tracked_frames
def detection_tracks_dataframe(tracked_frames):
    rows = []
    for frame_records in tracked_frames:
        for obj in frame_records:
            x1, y1, x2, y2 = obj["box"]
            rows.append({
                "frame": obj["frame"],
                "track_id": obj["track_id"],
                "class_name": obj["class_name"],
                "confidence": round(obj["confidence"], 3),
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
                "width": round(x2 - x1, 2),
                "height": round(y2 - y1, 2),
                "matched_iou": None if np.isnan(obj["matched_iou"]) else round(obj["matched_iou"], 3),
            })
    return pd.DataFrame(rows)
def show_detection_data(tracked_frames, title):
    df = detection_tracks_dataframe(tracked_frames)
    print(title)
    display(df.head(50))
    return df
def draw_yolo_tracked_frame(frame, frame_records):
    import cv2
    image = frame.copy()
    for obj in frame_records:
        x1, y1, x2, y2 = [int(v) for v in obj["box"]]
        label = f"T{obj['track_id']} {obj['class_name']} {obj['confidence']:.2f}"
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
    return image
def draw_yolo_frame(frame, frame_dets, conf_min=0.25):
    boxes, labels = yolo_dets(frame_dets, conf_min=conf_min)
    records = []
    for object_id, (box, label) in enumerate(zip(boxes, labels)):
        class_name, conf = label.rsplit(" ", 1)
        records.append({"track_id": object_id, "class_name": class_name, "confidence": float(conf), "box": box[:4]})
    return draw_yolo_tracked_frame(frame, records)
def show_detection_video(frames_batch, detections_batch, name, fps=8, clip_frames=48, seed=0):
    indices = clip_indices(min(len(frames_batch), len(detections_batch)), clip_frames=clip_frames, seed=seed)
    clip_detections = [detections_batch[i] for i in indices]
    clip_tracks = track_yolo_detections(clip_detections)
    annotated = [draw_yolo_tracked_frame(frames_batch[i], frame_records) for i, frame_records in zip(indices, clip_tracks)]
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    return show_video(np.asarray(annotated), name, fps=fps)


## 1 - Lunar Lander Detection


In [ ]:
# YOLO decides what, if anything, it recognizes in LunarLander. No manual class labels.
if "detections_lunar" not in globals():
    detections_lunar = run_yolo(frames)
if "lunar_yolo_counts" not in globals():
    lunar_yolo_counts = yolo_detection_counts(detections_lunar)
if "lunar_yolo_tracks" not in globals():
    lunar_yolo_tracks = track_yolo_detections(detections_lunar)
boxes, labels = yolo_dets(detections_lunar[frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames, detections_lunar, "1_lunar_lander_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(lunar_yolo_counts, "1 - YOLO detections per frame", ylabel="detections")
lunar_yolo_data = show_detection_data(lunar_yolo_tracks, "1 - Lunar Lander YOLO tracking data representation")


## 2 - Car Racing Detection


In [ ]:
# YOLO decides what, if anything, it recognizes in CarRacing. No manual class labels.
if "detections_car" not in globals():
    detections_car = run_yolo(frames_car)
if "car_yolo_counts" not in globals():
    car_yolo_counts = yolo_detection_counts(detections_car)
if "car_yolo_tracks" not in globals():
    car_yolo_tracks = track_yolo_detections(detections_car)
boxes, labels = yolo_dets(detections_car[car_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_car, detections_car, "2_car_racing_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(car_yolo_counts, "2 - YOLO detections per frame", ylabel="detections")
car_yolo_data = show_detection_data(car_yolo_tracks, "2 - Car Racing YOLO tracking data representation")


## 3 - Recorded Traffic With People Detection


In [ ]:
if "detections_traffic" not in globals():
    detections_traffic = run_yolo(frames_medium)
if "traffic_yolo_counts" not in globals():
    traffic_yolo_counts = yolo_detection_counts(detections_traffic)
if "traffic_yolo_tracks" not in globals():
    traffic_yolo_tracks = track_yolo_detections(detections_traffic)
det_frame_idx = min(10, len(frames_medium) - 1)
boxes, labels = yolo_dets(detections_traffic[det_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_medium, detections_traffic, "3_recorded_traffic_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(traffic_yolo_counts, "3 - YOLO detections per frame", ylabel="detections")
traffic_yolo_data = show_detection_data(traffic_yolo_tracks, "3 - Recorded Traffic YOLO tracking data representation")


## 4 - Random YouTube Driving Scene Detection


In [ ]:
if "detections_problem4" not in globals():
    detections_problem4 = run_yolo(frames_problem4)
if "problem4_yolo_counts" not in globals():
    problem4_yolo_counts = yolo_detection_counts(detections_problem4)
if "problem4_yolo_tracks" not in globals():
    problem4_yolo_tracks = track_yolo_detections(detections_problem4)
problem4_det_frame_idx = min(30, len(frames_problem4) - 1)
boxes, labels = yolo_dets(detections_problem4[problem4_det_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_problem4, detections_problem4, "4_youtube_driving_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(problem4_yolo_counts, "4 - YOLO detections per sampled frame", ylabel="detections")
problem4_yolo_data = show_detection_data(problem4_yolo_tracks, "4 - Random YouTube Driving YOLO tracking data representation")


## 5 - Hard Vehicle-Crowd Interaction Detection


In [ ]:
if "detections_problem5" not in globals():
    detections_problem5 = run_yolo(frames_problem5)
if "problem5_yolo_counts" not in globals():
    problem5_yolo_counts = yolo_detection_counts(detections_problem5)
if "problem5_yolo_tracks" not in globals():
    problem5_yolo_tracks = track_yolo_detections(detections_problem5)
problem5_det_frame_idx = min(40, len(frames_problem5) - 1)
boxes, labels = yolo_dets(detections_problem5[problem5_det_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_problem5, detections_problem5, "5_hard_vehicle_crowd_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(problem5_yolo_counts, "5 - YOLO detections per sampled frame", ylabel="detections")
problem5_yolo_data = show_detection_data(problem5_yolo_tracks, "5 - Hard Vehicle-Crowd YOLO tracking data representation")


# Object Segmentation


In [ ]:
# SAM-style class-agnostic object segmentation setup
# FastSAM returns mask proposals/blobs. We do not use semantic class labels here.
!pip install ultralytics -q
from ultralytics import FastSAM
import pandas as pd
seg_model = FastSAM("FastSAM-s.pt")
def run_segmentation(frame, imgsz=640, conf=0.35, iou=0.9):
    result = seg_model(frame, imgsz=imgsz, conf=conf, iou=iou, retina_masks=True, verbose=False)[0]
    if result.masks is None:
        return []
    masks = result.masks.data.cpu().numpy().astype(bool)
    return masks
def mask_to_record(mask, object_id):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    area = int(mask.sum())
    centroid = (float(xs.mean()), float(ys.mean()))
    return {
        "object_id": object_id,
        "mask": mask,
        "box": (x1, y1, x2, y2),
        "area": area,
        "centroid": centroid,
    }
def segmentation_records(frame, min_area=80, max_objects=25):
    masks = run_segmentation(frame)
    records = []
    for mask in masks:
        record = mask_to_record(mask, len(records))
        if record is not None and record["area"] >= min_area:
            records.append(record)
    records = sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
    for object_id, record in enumerate(records):
        record["object_id"] = object_id
    return records
def segmentation_summary(frames_batch, max_frames=40, stride=8, min_area=120):
    sampled = frames_batch[::stride][:max_frames]
    object_counts = []
    total_mask_area = []
    for frame in sampled:
        records = segmentation_records(frame, min_area=min_area)
        object_counts.append(len(records))
        total_mask_area.append(sum(obj["area"] for obj in records))
    return sampled, np.asarray(object_counts), np.asarray(total_mask_area)
def mask_boundary(mask):
    mask = mask.astype(bool)
    padded = np.pad(mask, 1, mode="constant", constant_values=False)
    center = padded[1:-1, 1:-1]
    eroded = (
        padded[:-2, 1:-1]
        & padded[2:, 1:-1]
        & padded[1:-1, :-2]
        & padded[1:-1, 2:]
        & center
    )
    return center & ~eroded
def mask_iou(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter / union) if union > 0 else 0.0
def box_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)
def box_area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)
def segmentation_match_score(track, record, stable_blob=False):
    # Match against the latest observed mask, but keep the first blob locked for output.
    track_mask = track.get("last_mask", track["mask"])
    track_box = track.get("last_box", track["box"])
    track_centroid = track.get("last_centroid", track["centroid"])
    track_area = track.get("last_area", track["area"])
    mask_score = mask_iou(track_mask, record["mask"])
    if not stable_blob:
        return mask_score, mask_score
    box_score = box_iou(track_box, record["box"])
    tx, ty = track_centroid
    rx, ry = record["centroid"]
    diag = max(1.0, np.hypot(*record["mask"].shape))
    centroid_score = max(0.0, 1.0 - (np.hypot(tx - rx, ty - ry) / (0.12 * diag)))
    area_ratio = min(track_area, record["area"]) / max(1.0, max(track_area, record["area"]))
    combined = 0.45 * mask_score + 0.25 * box_score + 0.2 * centroid_score + 0.1 * area_ratio
    return combined, mask_score
def apply_locked_blob(record, track):
    record["observed_mask"] = record["mask"]
    record["observed_box"] = record["box"]
    record["observed_centroid"] = record["centroid"]
    record["observed_area"] = record["area"]
    record["mask"] = track["locked_mask"]
    record["box"] = track["locked_box"]
    record["centroid"] = track["locked_centroid"]
    record["area"] = track["locked_area"]
    record["blob_locked"] = True
    return record
def track_segmentation_sequence(record_sequence, iou_threshold=0.25, max_missing=2, stable_blob=False):
    if stable_blob:
        iou_threshold = min(iou_threshold, 0.12)
        max_missing = max(max_missing, 8)
    tracked_sequence = []
    active_tracks = []
    next_track_id = 0
    for frame_no, records in enumerate(record_sequence):
        old_track_count = len(active_tracks)
        frame_records = [{**record, "frame": frame_no} for record in records]
        candidates = []
        used_tracks = set()
        used_records = set()
        for track_idx, track in enumerate(active_tracks):
            if track["missing"] > max_missing:
                continue
            for record_idx, record in enumerate(frame_records):
                score, mask_score = segmentation_match_score(track, record, stable_blob=stable_blob)
                threshold = 0.35 if stable_blob else iou_threshold
                if score >= threshold:
                    candidates.append((score, mask_score, track_idx, record_idx))
        for score, mask_score, track_idx, record_idx in sorted(candidates, reverse=True):
            if track_idx in used_tracks or record_idx in used_records:
                continue
            track = active_tracks[track_idx]
            record = frame_records[record_idx]
            record["track_id"] = track["track_id"]
            record["matched_iou"] = mask_score
            record["matched_score"] = score
            if stable_blob:
                apply_locked_blob(record, track)
                track.update({
                    "last_mask": record["observed_mask"],
                    "last_box": record["observed_box"],
                    "last_centroid": record["observed_centroid"],
                    "last_area": record["observed_area"],
                    "missing": 0,
                })
            else:
                track.update({
                    "mask": record["mask"],
                    "box": record["box"],
                    "centroid": record["centroid"],
                    "area": record["area"],
                    "missing": 0,
                })
            used_tracks.add(track_idx)
            used_records.add(record_idx)
        for record_idx, record in enumerate(frame_records):
            if record_idx in used_records:
                continue
            record["track_id"] = next_track_id
            record["matched_iou"] = np.nan
            record["matched_score"] = np.nan
            record["blob_locked"] = bool(stable_blob)
            active_tracks.append({
                "track_id": next_track_id,
                "mask": record["mask"],
                "box": record["box"],
                "centroid": record["centroid"],
                "area": record["area"],
                "locked_mask": record["mask"],
                "locked_box": record["box"],
                "locked_centroid": record["centroid"],
                "locked_area": record["area"],
                "last_mask": record["mask"],
                "last_box": record["box"],
                "last_centroid": record["centroid"],
                "last_area": record["area"],
                "missing": 0,
            })
            next_track_id += 1
        for track_idx, track in enumerate(active_tracks[:old_track_count]):
            if track_idx not in used_tracks:
                track["missing"] += 1
        active_tracks = [track for track in active_tracks if track["missing"] <= max_missing]
        tracked_sequence.append(sorted(frame_records, key=lambda obj: obj["track_id"]))
    return tracked_sequence
def show_segmentation_box_frame(frame, records, title):
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(frame)
    for record in records:
        x1, y1, x2, y2 = record["box"]
        cx, cy = record["centroid"]
        label_id = record.get("track_id", record["object_id"])
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=1.8, edgecolor="lime")
        ax.add_patch(rect)
        ax.scatter([cx], [cy], s=18, c="yellow", edgecolors="black", linewidths=0.6)
        ax.text(x1, max(0, y1 - 4), f"box_T{label_id}", color="lime", fontsize=8, weight="bold")
    ax.set_title(title)
    ax.axis("off")
    plt.show()
def show_segmentation_blob_frame(frame, records, title, alpha=0.45):
    overlay = frame.copy()
    colors = []
    for record in records:
        rng = np.random.default_rng(int(record.get("track_id", record["object_id"])) + 100)
        color = rng.integers(40, 255, size=3)
        colors.append(color)
        mask = record["mask"]
        overlay[mask] = (overlay[mask] * (1 - alpha) + color * alpha).astype(np.uint8)
        overlay[mask_boundary(mask)] = color
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(overlay)
    for record, color in zip(records, colors):
        cx, cy = record["centroid"]
        label_id = record.get("track_id", record["object_id"])
        ax.scatter([cx], [cy], s=18, c=[color / 255], edgecolors="black", linewidths=0.6)
        ax.text(cx + 3, cy + 3, f"blob_T{label_id}", color="white", fontsize=8, weight="bold")
    ax.set_title(title)
    ax.axis("off")
    plt.show()
def plot_segmentation_summary(object_counts, total_mask_area, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(object_counts)
    axes[0].set_title("mask count")
    axes[0].set_xlabel("sampled frame")
    axes[0].set_ylabel("objects")
    axes[0].grid(alpha=0.25)
    axes[1].plot(total_mask_area)
    axes[1].set_title("total mask area")
    axes[1].set_xlabel("sampled frame")
    axes[1].set_ylabel("pixels")
    axes[1].grid(alpha=0.25)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
def boundary_points(mask, max_points=24):
    boundary = mask_boundary(mask)
    ys, xs = np.nonzero(boundary)
    if len(xs) == 0:
        return []
    order = np.argsort(np.arctan2(ys - ys.mean(), xs - xs.mean()))
    xs, ys = xs[order], ys[order]
    if len(xs) > max_points:
        sample_idx = np.linspace(0, len(xs) - 1, max_points).astype(int)
        xs, ys = xs[sample_idx], ys[sample_idx]
    return [(int(x), int(y)) for x, y in zip(xs, ys)]
def segmentation_dataframe(records_or_sequence, representation="blob", max_boundary_points=24):
    if len(records_or_sequence) and isinstance(records_or_sequence[0], list):
        flat_records = [obj for frame_records in records_or_sequence for obj in frame_records]
    else:
        flat_records = list(records_or_sequence)
    rows = []
    for obj in flat_records:
        row = {
            "frame": obj.get("frame", 0),
            "track_id": obj.get("track_id", obj["object_id"]),
            "object_id": obj["object_id"],
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
            "matched_iou": None if np.isnan(obj.get("matched_iou", np.nan)) else round(obj["matched_iou"], 3),
            "matched_score": None if np.isnan(obj.get("matched_score", np.nan)) else round(obj["matched_score"], 3),
            "blob_locked": bool(obj.get("blob_locked", False)),
        }
        if representation == "box":
            x1, y1, x2, y2 = obj["box"]
            row.update({
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": x2 - x1,
                "height": y2 - y1,
            })
        elif representation == "blob":
            row.update({
                "mask_shape": obj["mask"].shape,
                "mask_pixels": int(obj["mask"].sum()),
                "boundary_sample": boundary_points(obj["mask"], max_points=max_boundary_points),
            })
        else:
            raise ValueError(representation)
        rows.append(row)
    return pd.DataFrame(rows)
def show_segmentation_data(records_or_sequence, representation, title):
    df = segmentation_dataframe(records_or_sequence, representation=representation)
    print(title)
    display(df.head(50))
    return df
def render_segmentation_frame(frame, records, view="blob", alpha=0.45):
    import cv2
    image = frame.copy()
    if view == "box":
        for record in records:
            x1, y1, x2, y2 = record["box"]
            cx, cy = record["centroid"]
            label_id = record.get("track_id", record["object_id"])
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(image, (int(cx), int(cy)), 3, (255, 255, 0), -1)
            cv2.putText(image, f"box_T{label_id}", (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
        return image
    for record in records:
        rng = np.random.default_rng(int(record.get("track_id", record["object_id"])) + 100)
        color = rng.integers(40, 255, size=3)
        mask = record["mask"]
        if mask.shape != image.shape[:2]:
            import cv2
            mask = cv2.resize(mask.astype(np.uint8), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
        image[mask] = (image[mask] * (1 - alpha) + color * alpha).astype(np.uint8)
        image[mask_boundary(mask)] = color
        cx, cy = record["centroid"]
        label_id = record.get("track_id", record["object_id"])
        cv2.circle(image, (int(cx), int(cy)), 3, (255, 255, 0), -1)
        cv2.putText(image, f"blob_T{label_id}", (int(cx) + 3, int(cy) + 3), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
    return image
def show_segmentation_video(frames_batch, name, view="blob", fps=4, clip_frames=24, seed=0, min_area=120):
    indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    raw_sequence = [segmentation_records(frames_batch[i], min_area=min_area) for i in indices]
    tracked_sequence = track_segmentation_sequence(raw_sequence, stable_blob=(view == "blob"))
    rendered = [
        render_segmentation_frame(frames_batch[i], records, view=view)
        for i, records in zip(indices, tracked_sequence)
    ]
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    return show_video(np.asarray(rendered), name, fps=fps), tracked_sequence
def show_segmentation_experiment(frames_batch, frame_idx, title, prefix, view="blob", min_area=120):
    if view not in {"box", "blob"}:
        raise ValueError(view)
    video_path, tracked_sequence = show_segmentation_video(frames_batch, f"{prefix}_{view}_segmentation.mp4", view=view, fps=4, clip_frames=24, seed=0, min_area=min_area)
    counts = np.asarray([len(records) for records in tracked_sequence], dtype=int)
    areas = np.asarray([sum(obj["area"] for obj in records) for records in tracked_sequence], dtype=int)
    print("tracked objects in clip:", len({obj["track_id"] for records in tracked_sequence for obj in records}))
    print([
        {"frame": obj["frame"], "track_id": obj["track_id"], "object_id": obj["object_id"], "area": obj["area"], "centroid": obj["centroid"]}
        for records in tracked_sequence[:3]
        for obj in records[:5]
    ][:10])
    plot_segmentation_summary(counts, areas, title)
    data = show_segmentation_data(tracked_sequence, representation=view, title=f"{title} tracking data representation")
    return tracked_sequence, None, counts, areas, data


## Object Segmentation Box


### 1 - Lunar Lander Box


In [ ]:
seg_box_1_lunar_tracks, seg_box_1_lunar_frames, seg_box_1_lunar_counts, seg_box_1_lunar_areas, seg_box_1_lunar_data = show_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Segmentation Box",
    "seg_box_1_lunar",
    view="box",
    min_area=120,
)


### 2 - Car Racing Box


In [ ]:
seg_box_2_car_tracks, seg_box_2_car_frames, seg_box_2_car_counts, seg_box_2_car_areas, seg_box_2_car_data = show_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Segmentation Box",
    "seg_box_2_car",
    view="box",
    min_area=120,
)


### 3 - Recorded Traffic With People Box


In [ ]:
seg_box_3_traffic_tracks, seg_box_3_traffic_frames, seg_box_3_traffic_counts, seg_box_3_traffic_areas, seg_box_3_traffic_data = show_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Segmentation Box",
    "seg_box_3_traffic",
    view="box",
    min_area=120,
)


### 4 - Random YouTube Driving Scene Box


In [ ]:
seg_box_4_youtube_tracks, seg_box_4_youtube_frames, seg_box_4_youtube_counts, seg_box_4_youtube_areas, seg_box_4_youtube_data = show_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Segmentation Box",
    "seg_box_4_youtube",
    view="box",
    min_area=120,
)


### 5 - Hard Vehicle-Crowd Interaction Box


In [ ]:
seg_box_5_hard_tracks, seg_box_5_hard_frames, seg_box_5_hard_counts, seg_box_5_hard_areas, seg_box_5_hard_data = show_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Segmentation Box",
    "seg_box_5_hard",
    view="box",
    min_area=120,
)


## Object Segmentation Blob (FastSAM)


### 1 - Lunar Lander Blob


In [ ]:
seg_blob_1_lunar_tracks, seg_blob_1_lunar_frames, seg_blob_1_lunar_counts, seg_blob_1_lunar_areas, seg_blob_1_lunar_data = show_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Segmentation Blob",
    "seg_blob_1_lunar",
    view="blob",
    min_area=120,
)


### 2 - Car Racing Blob


In [ ]:
seg_blob_2_car_tracks, seg_blob_2_car_frames, seg_blob_2_car_counts, seg_blob_2_car_areas, seg_blob_2_car_data = show_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Segmentation Blob",
    "seg_blob_2_car",
    view="blob",
    min_area=120,
)


### 3 - Recorded Traffic With People Blob


In [ ]:
seg_blob_3_traffic_tracks, seg_blob_3_traffic_frames, seg_blob_3_traffic_counts, seg_blob_3_traffic_areas, seg_blob_3_traffic_data = show_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Segmentation Blob",
    "seg_blob_3_traffic",
    view="blob",
    min_area=120,
)


### 4 - Random YouTube Driving Scene Blob


In [ ]:
seg_blob_4_youtube_tracks, seg_blob_4_youtube_frames, seg_blob_4_youtube_counts, seg_blob_4_youtube_areas, seg_blob_4_youtube_data = show_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Segmentation Blob",
    "seg_blob_4_youtube",
    view="blob",
    min_area=120,
)


### 5 - Hard Vehicle-Crowd Interaction Blob


In [ ]:
seg_blob_5_hard_tracks, seg_blob_5_hard_frames, seg_blob_5_hard_counts, seg_blob_5_hard_areas, seg_blob_5_hard_data = show_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Segmentation Blob",
    "seg_blob_5_hard",
    view="blob",
    min_area=120,
)


## Object Segmentation Blob (YOLO11-seg)


In [ ]:
# YOLO11-seg object segmentation blob setup
# This variant segments known active-object classes instead of arbitrary scene blobs.
!pip install ultralytics opencv-python -q
from ultralytics import YOLO
yolo_seg_model = YOLO("models/yolov8n-seg.pt")
YOLO_SEG_ACTIVE_CLASS_IDS = {0, 1, 2, 3, 5, 7}  # person, bicycle, car, motorcycle, bus, truck
def yolo_segmentation_records(frame, conf_min=0.25, max_objects=40):
    result = yolo_seg_model(frame, verbose=False, conf=conf_min)[0]
    records = []
    if result.masks is None:
        return records
    masks = result.masks.data.cpu().numpy().astype(bool)
    if masks.shape[1:] != frame.shape[:2]:
        import cv2
        masks = np.asarray([
            cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
            for mask in masks
        ])
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()
    for mask, class_id, conf in zip(masks, classes, confs):
        if int(class_id) not in YOLO_SEG_ACTIVE_CLASS_IDS or float(conf) < conf_min:
            continue
        record = mask_to_record(mask, len(records))
        if record is None:
            continue
        record["class_id"] = int(class_id)
        record["class_name"] = result.names[int(class_id)]
        record["confidence"] = float(conf)
        records.append(record)
    return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
def show_yolo_blob_segmentation_video(frames_batch, name, fps=4, clip_frames=24, seed=0, conf_min=0.25):
    indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    raw_sequence = [yolo_segmentation_records(frames_batch[i], conf_min=conf_min) for i in indices]
    tracked_sequence = track_segmentation_sequence(raw_sequence, stable_blob=True)
    rendered = [render_segmentation_frame(frames_batch[i], records, view="blob") for i, records in zip(indices, tracked_sequence)]
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    return show_video(np.asarray(rendered), name, fps=fps), tracked_sequence
def show_yolo_blob_segmentation_experiment(frames_batch, frame_idx, title, prefix, conf_min=0.25):
    video_path, tracked_sequence = show_yolo_blob_segmentation_video(frames_batch, f"{prefix}_yolo11seg_blob_segmentation.mp4", fps=4, clip_frames=24, seed=0, conf_min=conf_min)
    counts = np.asarray([len(records) for records in tracked_sequence], dtype=int)
    areas = np.asarray([sum(obj["area"] for obj in records) for records in tracked_sequence], dtype=int)
    print("tracked objects in clip:", len({obj["track_id"] for records in tracked_sequence for obj in records}))
    plot_segmentation_summary(counts, areas, title)
    data = show_segmentation_data(tracked_sequence, representation="blob", title=f"{title} tracking data representation")
    return tracked_sequence, None, counts, areas, data


### 1 - Lunar Lander Blob


In [ ]:
seg_blob_yolo11seg_1_lunar_tracks, seg_blob_yolo11seg_1_lunar_frames, seg_blob_yolo11seg_1_lunar_counts, seg_blob_yolo11seg_1_lunar_areas, seg_blob_yolo11seg_1_lunar_data = show_yolo_blob_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Segmentation Blob (YOLO11-seg)",
    "seg_blob_yolo11seg_1_lunar",
)


### 2 - Car Racing Blob


In [ ]:
seg_blob_yolo11seg_2_car_tracks, seg_blob_yolo11seg_2_car_frames, seg_blob_yolo11seg_2_car_counts, seg_blob_yolo11seg_2_car_areas, seg_blob_yolo11seg_2_car_data = show_yolo_blob_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Segmentation Blob (YOLO11-seg)",
    "seg_blob_yolo11seg_2_car",
)


### 3 - Recorded Traffic With People Blob


In [ ]:
seg_blob_yolo11seg_3_traffic_tracks, seg_blob_yolo11seg_3_traffic_frames, seg_blob_yolo11seg_3_traffic_counts, seg_blob_yolo11seg_3_traffic_areas, seg_blob_yolo11seg_3_traffic_data = show_yolo_blob_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Segmentation Blob (YOLO11-seg)",
    "seg_blob_yolo11seg_3_traffic",
)


### 4 - Random YouTube Driving Scene Blob


In [ ]:
seg_blob_yolo11seg_4_youtube_tracks, seg_blob_yolo11seg_4_youtube_frames, seg_blob_yolo11seg_4_youtube_counts, seg_blob_yolo11seg_4_youtube_areas, seg_blob_yolo11seg_4_youtube_data = show_yolo_blob_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Segmentation Blob (YOLO11-seg)",
    "seg_blob_yolo11seg_4_youtube",
)


### 5 - Hard Vehicle-Crowd Interaction Blob


In [ ]:
seg_blob_yolo11seg_5_hard_tracks, seg_blob_yolo11seg_5_hard_frames, seg_blob_yolo11seg_5_hard_counts, seg_blob_yolo11seg_5_hard_areas, seg_blob_yolo11seg_5_hard_data = show_yolo_blob_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Segmentation Blob (YOLO11-seg)",
    "seg_blob_yolo11seg_5_hard",
)


## Object Segmentation Blob (YOLOv8-seg)


In [ ]:
# YOLOv8-seg object segmentation blob setup
# Model from the referenced Ikomia article: YOLOv8-seg, real-time instance segmentation.
!pip install ultralytics opencv-python -q
from ultralytics import YOLO
yolov8_seg_model = YOLO("models/yolov8n-seg.pt")
YOLOV8_SEG_ACTIVE_CLASS_IDS = {0, 1, 2, 3, 5, 7}  # person, bicycle, car, motorcycle, bus, truck
def yolov8_segmentation_records(frame, conf_min=0.25, max_objects=40):
    result = yolov8_seg_model(frame, verbose=False, conf=conf_min)[0]
    records = []
    if result.masks is None:
        return records
    masks = result.masks.data.cpu().numpy().astype(bool)
    if masks.shape[1:] != frame.shape[:2]:
        import cv2
        masks = np.asarray([
            cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
            for mask in masks
        ])
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()
    for mask, class_id, conf in zip(masks, classes, confs):
        if int(class_id) not in YOLOV8_SEG_ACTIVE_CLASS_IDS or float(conf) < conf_min:
            continue
        record = mask_to_record(mask, len(records))
        if record is None:
            continue
        record["class_id"] = int(class_id)
        record["class_name"] = result.names[int(class_id)]
        record["confidence"] = float(conf)
        records.append(record)
    return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
def show_yolov8_blob_segmentation_video(frames_batch, name, fps=4, clip_frames=24, seed=0, conf_min=0.25):
    indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    raw_sequence = [yolov8_segmentation_records(frames_batch[i], conf_min=conf_min) for i in indices]
    tracked_sequence = track_segmentation_sequence(raw_sequence, stable_blob=True)
    rendered = [render_segmentation_frame(frames_batch[i], records, view="blob") for i, records in zip(indices, tracked_sequence)]
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    return show_video(np.asarray(rendered), name, fps=fps), tracked_sequence
def show_yolov8_blob_segmentation_experiment(frames_batch, frame_idx, title, prefix, conf_min=0.25):
    video_path, tracked_sequence = show_yolov8_blob_segmentation_video(frames_batch, f"{prefix}_yolov8seg_blob_segmentation.mp4", fps=4, clip_frames=24, seed=0, conf_min=conf_min)
    counts = np.asarray([len(records) for records in tracked_sequence], dtype=int)
    areas = np.asarray([sum(obj["area"] for obj in records) for records in tracked_sequence], dtype=int)
    print("tracked objects in clip:", len({obj["track_id"] for records in tracked_sequence for obj in records}))
    plot_segmentation_summary(counts, areas, title)
    data = show_segmentation_data(tracked_sequence, representation="blob", title=f"{title} tracking data representation")
    return tracked_sequence, None, counts, areas, data


### 1 - Lunar Lander Blob


In [ ]:
seg_blob_yolov8seg_1_lunar_tracks, seg_blob_yolov8seg_1_lunar_frames, seg_blob_yolov8seg_1_lunar_counts, seg_blob_yolov8seg_1_lunar_areas, seg_blob_yolov8seg_1_lunar_data = show_yolov8_blob_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Segmentation Blob (YOLOv8-seg)",
    "seg_blob_yolov8seg_1_lunar",
)


### 2 - Car Racing Blob


In [ ]:
seg_blob_yolov8seg_2_car_tracks, seg_blob_yolov8seg_2_car_frames, seg_blob_yolov8seg_2_car_counts, seg_blob_yolov8seg_2_car_areas, seg_blob_yolov8seg_2_car_data = show_yolov8_blob_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Segmentation Blob (YOLOv8-seg)",
    "seg_blob_yolov8seg_2_car",
)


### 3 - Recorded Traffic With People Blob


In [ ]:
seg_blob_yolov8seg_3_traffic_tracks, seg_blob_yolov8seg_3_traffic_frames, seg_blob_yolov8seg_3_traffic_counts, seg_blob_yolov8seg_3_traffic_areas, seg_blob_yolov8seg_3_traffic_data = show_yolov8_blob_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Segmentation Blob (YOLOv8-seg)",
    "seg_blob_yolov8seg_3_traffic",
)


### 4 - Random YouTube Driving Scene Blob


In [ ]:
seg_blob_yolov8seg_4_youtube_tracks, seg_blob_yolov8seg_4_youtube_frames, seg_blob_yolov8seg_4_youtube_counts, seg_blob_yolov8seg_4_youtube_areas, seg_blob_yolov8seg_4_youtube_data = show_yolov8_blob_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Segmentation Blob (YOLOv8-seg)",
    "seg_blob_yolov8seg_4_youtube",
)


### 5 - Hard Vehicle-Crowd Interaction Blob


In [ ]:
seg_blob_yolov8seg_5_hard_tracks, seg_blob_yolov8seg_5_hard_frames, seg_blob_yolov8seg_5_hard_counts, seg_blob_yolov8seg_5_hard_areas, seg_blob_yolov8seg_5_hard_data = show_yolov8_blob_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Segmentation Blob (YOLOv8-seg)",
    "seg_blob_yolov8seg_5_hard",
)


## WINNER - Object Segmentation Blob (Active Agents Only: People + Vehicles)


In [ ]:
# Minimal standalone loader for this section.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw
try:
    import cv2
except Exception as exc:
    cv2 = None
    print(f"cv2 unavailable: {type(exc).__name__}: {exc}")
def load_video_frames_minimal(path, max_frames=120, stride=3, start_seconds=0):
    path = Path(path)
    if cv2 is not None:
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_seconds * fps))
        out = []
        frame_no = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_no % stride == 0:
                out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if len(out) >= max_frames:
                    break
            frame_no += 1
        cap.release()
        if out:
            return np.stack(out)
    reader = imageio.get_reader(path)
    out = []
    for frame_no, frame in enumerate(reader):
        if frame_no % stride == 0:
            out.append(np.asarray(frame)[..., :3])
            if len(out) >= max_frames:
                break
    reader.close()
    if not out:
        raise RuntimeError(f"No frames loaded from {path}")
    return np.stack(out)
def ensure_frames_var(var_name, path, idx_name, idx_value, max_frames=120, stride=3, start_seconds=0):
    if var_name not in globals():
        globals()[var_name] = load_video_frames_minimal(path, max_frames=max_frames, stride=stride, start_seconds=start_seconds)
        print(f"loaded {var_name}:", globals()[var_name].shape)
    if idx_name not in globals():
        globals()[idx_name] = min(idx_value, len(globals()[var_name]) - 1)
        print(f"set {idx_name}:", globals()[idx_name])
ensure_frames_var("frames", "generated_videos/1_lunar_lander.gif", "frame_idx", 10, max_frames=80, stride=1)
ensure_frames_var("frames_car", "generated_videos/2_car_racing.gif", "car_frame_idx", 40, max_frames=120, stride=1)
ensure_frames_var("frames_medium", "external_videos/traffic.avi", "traffic_frame_idx", 1, max_frames=120, stride=1)
ensure_frames_var("frames_problem4", "external_videos/problem_4_youtube_random.mp4", "problem4_frame_idx", 30, max_frames=120, stride=3, start_seconds=20)
ensure_frames_var("frames_problem5", "external_videos/problem_5_youtube_hardest.mp4", "problem5_frame_idx", 40, max_frames=120, stride=3, start_seconds=20)
from ultralytics import YOLO
active_agent_seg_model = YOLO("models/yolov8n-seg.pt")
VIDEO_DIR = PROJECT_ROOT / "generated_videos"
VIDEO_DIR.mkdir(exist_ok=True)
def save_video(frames, path, fps=20):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = frames.astype(np.uint8)
    try:
        imageio.mimsave(path, frames, fps=fps)
        return path
    except Exception as exc:
        fallback = path.with_suffix(".gif")
        print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
        print(f"Saving GIF fallback: {fallback}")
        imageio.mimsave(fallback, frames, duration=1 / fps)
        return fallback
def show_video(frames, name, fps=20, embed=True):
    from IPython.display import Image as DisplayImage, Video
    path = save_video(frames, VIDEO_DIR / name, fps=fps)
    if path.suffix.lower() == ".gif":
        display(DisplayImage(filename=str(path)))
    else:
        display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def clip_indices(num_frames, clip_frames=48, seed=0):
    if num_frames <= 0:
        return np.asarray([], dtype=int)
    clip_frames = min(int(clip_frames), int(num_frames))
    rng = np.random.default_rng(seed)
    max_start = max(0, int(num_frames) - clip_frames)
    start = int(rng.integers(0, max_start + 1)) if max_start else 0
    return np.arange(start, start + clip_frames, dtype=int)
def mask_to_record(mask, object_id):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    area = int(mask.sum())
    centroid = (float(xs.mean()), float(ys.mean()))
    return {
        "object_id": object_id,
        "mask": mask,
        "box": (x1, y1, x2, y2),
        "area": area,
        "centroid": centroid,
    }
# Active-agent-only segmentation: people + vehicles.
# Background is everything outside these selected class masks.
ACTIVE_AGENT_CLASS_NAMES = {
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"
}
def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
    result = model(frame, verbose=False, conf=conf_min)[0]
    records = []
    if result.masks is None:
        return records
    masks = result.masks.data.cpu().numpy().astype(bool)
    if masks.shape[1:] != frame.shape[:2]:
        import cv2
        masks = np.asarray([
            cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
            for mask in masks
        ])
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()
    for mask, class_id, conf in zip(masks, classes, confs):
        class_name = result.names[int(class_id)]
        if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
            continue
        record = mask_to_record(mask, len(records))
        if record is None:
            continue
        record["class_id"] = int(class_id)
        record["class_name"] = class_name
        record["confidence"] = float(conf)
        records.append(record)
    return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
def render_active_agents_inverse_background(frame, records, title, alpha_bg=0.35, alpha_agent=0.55):
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    bg_mask = ~active_mask
    image = frame.copy()
    image[bg_mask] = (image[bg_mask] * (1 - alpha_bg) + np.array([40, 180, 90]) * alpha_bg).astype(np.uint8)
    image[active_mask] = (image[active_mask] * (1 - alpha_agent) + np.array([230, 50, 40]) * alpha_agent).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()
    return bg_mask, active_mask
def show_active_agent_only_segmentation_experiment(frames_batch, frame_idx, title, model=None, conf_min=0.25):
    if model is None:
        model = active_agent_seg_model
    frame = frames_batch[frame_idx]
    records = active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min)
    bg_mask, active_mask = render_active_agents_inverse_background(frame, records, title)
    rows = []
    for obj in records:
        rows.append({
            "object_id": obj["object_id"],
            "class_name": obj["class_name"],
            "confidence": round(obj["confidence"], 3),
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
            "box": obj["box"],
        })
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "selected_classes": sorted(ACTIVE_AGENT_CLASS_NAMES),
        "agents": len(records),
        "active_agent_pixels": int(active_mask.sum()),
        "background_pixels": int(bg_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
    }])
    display(summary)
    display(pd.DataFrame(rows))
    return bg_mask, records, summary
def save_active_agent_video(frames_batch, name, model=None, fps=6, clip_frames=36, seed=0, conf_min=0.25):
    if model is None:
        model = active_agent_seg_model
    (PROJECT_ROOT / "generated_videos").mkdir(exist_ok=True)
    indices = np.arange(min(len(frames_batch), clip_frames), dtype=int)
    if len(frames_batch) > clip_frames:
        indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed) if "clip_indices" in globals() else indices
    rendered = []
    raw_sequence = []
    for i in indices:
        records = active_agent_records_from_yolo_seg(frames_batch[i], model=model, conf_min=conf_min)
        raw_sequence.append(records)
        active_mask = np.zeros(frames_batch[i].shape[:2], dtype=bool)
        for record in records:
            active_mask |= record["mask"]
        bg_mask = ~active_mask
        image = frames_batch[i].copy()
        image[bg_mask] = (image[bg_mask] * 0.65 + np.array([40, 180, 90]) * 0.35).astype(np.uint8)
        image[active_mask] = (image[active_mask] * 0.45 + np.array([230, 50, 40]) * 0.55).astype(np.uint8)
        rendered.append(image)
    tracked_sequence = track_segmentation_sequence(raw_sequence, stable_blob=True) if "track_segmentation_sequence" in globals() else raw_sequence
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    path = show_video(np.asarray(rendered), name, fps=fps) if "show_video" in globals() else None
    return path, tracked_sequence
def show_active_agent_only_segmentation_experiment(frames_batch, frame_idx, title, model=None, conf_min=0.25, video_name=None):
    if model is None:
        model = active_agent_seg_model
    frame = frames_batch[frame_idx]
    records = active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min)
    bg_mask, active_mask = render_active_agents_inverse_background(frame, records, title)
    video_path = None
    video_tracks = None
    if video_name is not None:
        video_path, video_tracks = save_active_agent_video(frames_batch, video_name, model=model, conf_min=conf_min)
    rows = []
    for obj in records:
        rows.append({
            "object_id": obj["object_id"],
            "class_name": obj["class_name"],
            "confidence": round(obj["confidence"], 3),
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
            "box": obj["box"],
        })
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "selected_classes": sorted(ACTIVE_AGENT_CLASS_NAMES),
        "agents": len(records),
        "active_agent_pixels": int(active_mask.sum()),
        "background_pixels": int(bg_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
        "video_path": str(video_path) if video_path is not None else None,
    }])
    display(summary)
    display(pd.DataFrame(rows))
    return bg_mask, records, summary


### 1 - Lunar Lander


In [ ]:
active_agents_bg_1_lunar_mask, active_agents_1_lunar_records, active_agents_1_lunar_data = show_active_agent_only_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_1_lunar.mp4"
)


### 2 - Car Racing


In [ ]:
active_agents_bg_2_car_mask, active_agents_2_car_records, active_agents_2_car_data = show_active_agent_only_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_2_car.mp4"
)


### 3 - Recorded Traffic With People


In [ ]:
active_agents_bg_3_traffic_mask, active_agents_3_traffic_records, active_agents_3_traffic_data = show_active_agent_only_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_3_traffic.mp4"
)


### 4 - Random YouTube Driving Scene


In [ ]:
active_agents_bg_4_youtube_mask, active_agents_4_youtube_records, active_agents_4_youtube_data = show_active_agent_only_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_4_youtube.mp4"
)


### 5 - Hard Vehicle-Crowd Interaction


In [ ]:
active_agents_bg_5_hard_mask, active_agents_5_hard_records, active_agents_5_hard_data = show_active_agent_only_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_5_hard.mp4"
)


# Background Segmentation


Goal: mark background pixels/regions without active agents.


## Background Base Grid (Active-Agent Inverse + Lower Image)


In [ ]:
# Background base grid from Active Agents Only inverse mask.
# This is the first practical base-map approximation: use non-agent pixels in the lower image region.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw
try:
    import cv2
except Exception as exc:
    cv2 = None
    print(f"cv2 unavailable: {type(exc).__name__}: {exc}")

def base_grid_agent_records(frame, conf_min=0.25):
    if "active_agent_records_from_yolo_seg" in globals():
        model = active_agent_seg_model if "active_agent_seg_model" in globals() else None
        if model is not None:
            return active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min)
    return []

def agent_inverse_mask(frame, records):
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    return ~active_mask, active_mask

def lower_base_mask(frame, background_mask, y_start_ratio=0.55):
    h, w = frame.shape[:2]
    lower = np.zeros((h, w), dtype=bool)
    lower[int(h * y_start_ratio):, :] = True
    return background_mask & lower

def make_base_grid_points(base_mask, step=40, margin=20):
    h, w = base_mask.shape
    points = []
    for y in range(margin, h - margin, step):
        for x in range(margin, w - margin, step):
            if base_mask[y, x]:
                points.append((x, y))
    return np.asarray(points, dtype=np.float32)

def track_base_grid_points(frame_t, frame_t1, points):
    if cv2 is None or len(points) == 0:
        return pd.DataFrame(columns=["point_id", "x_t", "y_t", "x_t1", "y_t1", "dx", "dy", "speed"])
    gray_t = cv2.cvtColor(frame_t, cv2.COLOR_RGB2GRAY)
    gray_t1 = cv2.cvtColor(frame_t1, cv2.COLOR_RGB2GRAY)
    pts = points.reshape(-1, 1, 2).astype(np.float32)
    pts1, status, _ = cv2.calcOpticalFlowPyrLK(gray_t, gray_t1, pts, None, winSize=(21,21), maxLevel=3)
    status = status.reshape(-1).astype(bool)
    p0 = pts.reshape(-1, 2)[status]
    p1 = pts1.reshape(-1, 2)[status]
    flow = p1 - p0
    speed = np.linalg.norm(flow, axis=1)
    return pd.DataFrame({
        "point_id": np.arange(len(p0)),
        "x_t": p0[:,0], "y_t": p0[:,1],
        "x_t1": p1[:,0], "y_t1": p1[:,1],
        "dx": flow[:,0], "dy": flow[:,1],
        "speed": speed,
    })

def render_base_grid(frame, base_mask, active_mask, points, flow_df=None, title="Background base grid"):
    image = frame.copy()
    image[base_mask] = (image[base_mask] * 0.55 + np.array([40, 180, 90]) * 0.45).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.45 + np.array([230, 50, 40]) * 0.55).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    if len(points):
        plt.scatter(points[:,0], points[:,1], s=12, c="yellow", edgecolors="black", linewidths=0.3)
    if flow_df is not None and len(flow_df):
        plt.quiver(flow_df["x_t"], flow_df["y_t"], flow_df["dx"], flow_df["dy"], color="cyan", angles="xy", scale_units="xy", scale=1, width=0.003)
    plt.title(title)
    plt.axis("off")
    plt.show()

def show_background_base_grid_experiment(frames_batch, frame_idx, title, y_start_ratio=0.55, step=40):
    frame = frames_batch[frame_idx]
    next_idx = min(frame_idx + 1, len(frames_batch) - 1)
    records = base_grid_agent_records(frame)
    background_mask, active_mask = agent_inverse_mask(frame, records)
    base_mask = lower_base_mask(frame, background_mask, y_start_ratio=y_start_ratio)
    points = make_base_grid_points(base_mask, step=step)
    flow_df = track_base_grid_points(frame, frames_batch[next_idx], points)
    render_base_grid(frame, base_mask, active_mask, points, flow_df=flow_df, title=title)
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "base_region": "lower image background inverse-agent mask",
        "agents": len(records),
        "grid_points": int(len(points)),
        "tracked_points": int(len(flow_df)),
        "median_dx": float(flow_df["dx"].median()) if len(flow_df) else 0.0,
        "median_dy": float(flow_df["dy"].median()) if len(flow_df) else 0.0,
        "median_speed": float(flow_df["speed"].median()) if len(flow_df) else 0.0,
    }])
    display(summary)
    display(flow_df.head(30))
    return base_mask, points, flow_df, summary



### 1 - Lunar Lander Background Base Grid


In [ ]:
bg_base_grid_1_lunar_mask, bg_base_grid_1_lunar_points, bg_base_grid_1_lunar_flow, bg_base_grid_1_lunar_data = show_background_base_grid_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Background Base Grid"
)


### 2 - Car Racing Background Base Grid


In [ ]:
bg_base_grid_2_car_mask, bg_base_grid_2_car_points, bg_base_grid_2_car_flow, bg_base_grid_2_car_data = show_background_base_grid_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Background Base Grid"
)


### 3 - Recorded Traffic With People Background Base Grid


In [ ]:
bg_base_grid_3_traffic_mask, bg_base_grid_3_traffic_points, bg_base_grid_3_traffic_flow, bg_base_grid_3_traffic_data = show_background_base_grid_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Background Base Grid"
)


### 4 - Random YouTube Driving Scene Background Base Grid


In [ ]:
bg_base_grid_4_youtube_mask, bg_base_grid_4_youtube_points, bg_base_grid_4_youtube_flow, bg_base_grid_4_youtube_data = show_background_base_grid_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Background Base Grid"
)


### 5 - Hard Vehicle-Crowd Interaction Background Base Grid


In [ ]:
bg_base_grid_5_hard_mask, bg_base_grid_5_hard_points, bg_base_grid_5_hard_flow, bg_base_grid_5_hard_data = show_background_base_grid_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Background Base Grid"
)


## Winner - Background Surface Grid Tracking (Active-Agent Inverse)

In [ ]:
# Background surface grid tracking from Active Agents Only inverse mask.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
!pip install "imageio[ffmpeg]" -q

def ensure_surface_grid_dependencies():
    global active_agent_seg_model
    if "active_agent_seg_model" not in globals():
        from ultralytics import YOLO
        active_agent_seg_model = YOLO("models/yolov8n-seg.pt")
    if "ACTIVE_AGENT_CLASS_NAMES" not in globals():
        globals()["ACTIVE_AGENT_CLASS_NAMES"] = {"person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"}
    if "mask_to_record" not in globals():
        def mask_to_record(mask, object_id):
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                return None
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            return {"object_id": object_id, "mask": mask, "box": (x1, y1, x2, y2), "area": int(mask.sum()), "centroid": (float(xs.mean()), float(ys.mean()))}
        globals()["mask_to_record"] = mask_to_record
    if "active_agent_records_from_yolo_seg" not in globals():
        def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
            result = model(frame, verbose=False, conf=conf_min)[0]
            records = []
            if result.masks is None:
                return records
            masks = result.masks.data.cpu().numpy().astype(bool)
            if masks.shape[1:] != frame.shape[:2]:
                import cv2
                masks = np.asarray([cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool) for mask in masks])
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()
            for mask, class_id, conf in zip(masks, classes, confs):
                class_name = result.names[int(class_id)]
                if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
                    continue
                record = mask_to_record(mask, len(records))
                if record is None:
                    continue
                record["class_id"] = int(class_id)
                record["class_name"] = class_name
                record["confidence"] = float(conf)
                records.append(record)
            return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
        globals()["active_agent_records_from_yolo_seg"] = active_agent_records_from_yolo_seg
    if "clip_indices" not in globals():
        def clip_indices(num_frames, clip_frames=48, seed=0):
            if num_frames <= 0:
                return np.asarray([], dtype=int)
            clip_frames = min(int(clip_frames), int(num_frames))
            rng = np.random.default_rng(seed)
            max_start = max(0, int(num_frames) - clip_frames)
            start = int(rng.integers(0, max_start + 1)) if max_start else 0
            return np.arange(start, start + clip_frames, dtype=int)
        globals()["clip_indices"] = clip_indices
    if "show_video" not in globals():
        VIDEO_DIR = PROJECT_ROOT / "generated_videos"
        VIDEO_DIR.mkdir(exist_ok=True)
        def save_video(frames, path, fps=20):
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            frames = frames.astype(np.uint8)
            try:
                imageio.mimsave(path, frames, fps=fps)
                return path
            except Exception as exc:
                fallback = path.with_suffix(".gif")
                print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
                print(f"Saving GIF fallback: {fallback}")
                imageio.mimsave(fallback, frames, duration=1 / fps)
                return fallback
        def show_video(frames, name, fps=20, embed=True):
            from IPython.display import Image as DisplayImage, Video
            path = save_video(frames, VIDEO_DIR / name, fps=fps)
            if path.suffix.lower() == ".gif":
                display(DisplayImage(filename=str(path)))
            else:
                display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
            return path
        globals()["show_video"] = show_video
    if "read_video" not in globals():
        def read_video(path, max_frames=None, stride=1, start_frame=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            if start_frame:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
            frames_out = []
            frame_no = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_no % stride == 0:
                    frames_out.append(frame[:, :, ::-1])
                    if max_frames is not None and len(frames_out) >= max_frames:
                        break
                frame_no += 1
            cap.release()
            if not frames_out:
                raise RuntimeError(f"No frames read from {path}")
            return np.stack(frames_out)
        globals()["read_video"] = read_video
    if "read_video_sample" not in globals():
        def read_video_sample(path, max_frames=120, stride=3, start_seconds=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            cap.release()
            return read_video(path, max_frames=max_frames, stride=stride, start_frame=int(start_seconds * fps))
        globals()["read_video_sample"] = read_video_sample
    if "frames" not in globals():
        try:
            import gymnasium as gym
            from gymnasium.envs.box2d.lunar_lander import heuristic
            env = gym.make("LunarLander-v3", render_mode="rgb_array")
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(80):
                action = int(heuristic(env.unwrapped, obs))
                obs, _, terminated, truncated, _ = env.step(action)
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames"] = np.stack(out)
            globals()["frame_idx"] = min(10, len(frames) - 1)
        except Exception as exc:
            print(f"Lunar standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_car" not in globals():
        try:
            import gymnasium as gym
            env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(120):
                obs, _, terminated, truncated, _ = env.step(env.action_space.sample())
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames_car"] = np.stack(out)
            globals()["car_frame_idx"] = min(40, len(frames_car) - 1)
        except Exception as exc:
            print(f"Car standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_medium" not in globals() and PROJECT_ROOT / "external_videos/traffic.avi").exists():
        globals()["frames_medium"] = read_video("external_videos/traffic.avi", max_frames=100, stride=1)
        globals()["traffic_frame_idx"] = min(1, len(frames_medium) - 1)
    if "frames_problem4" not in globals() and PROJECT_ROOT / "external_videos/problem_4_youtube_random.mp4").exists():
        globals()["frames_problem4"] = read_video_sample("external_videos/problem_4_youtube_random.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem4_frame_idx"] = min(30, len(frames_problem4) - 1)
    if "frames_problem5" not in globals() and PROJECT_ROOT / "external_videos/problem_5_youtube_hardest.mp4").exists():
        globals()["frames_problem5"] = read_video_sample("external_videos/problem_5_youtube_hardest.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem5_frame_idx"] = min(40, len(frames_problem5) - 1)

def surface_grid_background_mask(frame, conf_min=0.25, dilate_px=7):
    import cv2
    ensure_surface_grid_dependencies()
    records = active_agent_records_from_yolo_seg(frame, model=active_agent_seg_model, conf_min=conf_min)
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    if dilate_px > 0 and active_mask.any():
        kernel = np.ones((dilate_px, dilate_px), dtype=np.uint8)
        active_mask = cv2.dilate(active_mask.astype(np.uint8), kernel, iterations=1).astype(bool)
    return ~active_mask, active_mask, records

def make_surface_grid_points(mask, step=32, margin=12):
    h, w = mask.shape
    pts = []
    for y in range(margin, h - margin, step):
        for x in range(margin, w - margin, step):
            if mask[y, x]:
                pts.append((float(x), float(y)))
    return np.asarray(pts, dtype=np.float32)

def add_missing_surface_grid_points(bg_mask, points, valid, step=32, min_distance_ratio=0.72):
    candidates = make_surface_grid_points(bg_mask, step=step)
    if len(candidates) == 0:
        return points.reshape(-1, 2), valid
    points = points.reshape(-1, 2)
    valid = valid.astype(bool)
    existing = points[valid] if len(points) else np.empty((0, 2), dtype=np.float32)
    new_points = []
    min_dist_sq = float(step * min_distance_ratio) ** 2
    for candidate in candidates:
        if len(existing):
            nearest_existing = np.min(np.sum((existing - candidate) ** 2, axis=1))
            if nearest_existing < min_dist_sq:
                continue
        if new_points:
            added = np.asarray(new_points, dtype=np.float32)
            nearest_added = np.min(np.sum((added - candidate) ** 2, axis=1))
            if nearest_added < min_dist_sq:
                continue
        new_points.append(candidate)
    if not new_points:
        return points, valid
    points = np.vstack([points, np.asarray(new_points, dtype=np.float32)]) if len(points) else np.asarray(new_points, dtype=np.float32)
    valid = np.concatenate([valid, np.ones(len(new_points), dtype=bool)])
    return points, valid

def track_surface_grid_points(frames_batch, indices, step=32):
    import cv2
    first = frames_batch[int(indices[0])]
    bg_mask, active_mask, records = surface_grid_background_mask(first)
    points0 = make_surface_grid_points(bg_mask, step=step)
    tracks = [{"frame": int(indices[0]), "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}]
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    for idx in indices[1:]:
        frame = frames_batch[int(idx)]
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        bg_mask, active_mask, records = surface_grid_background_mask(frame)
        if len(prev_pts) == 0:
            next_pts = prev_pts.copy()
            valid = np.zeros_like(valid)
        else:
            next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
            if next_pts is None or status is None:
                valid = np.zeros_like(valid)
                next_pts = prev_pts.copy()
            else:
                next_xy = next_pts.reshape(-1, 2)
                h, w = bg_mask.shape
                inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                on_background = np.zeros(len(next_xy), dtype=bool)
                rounded = np.floor(next_xy[inside]).astype(int)
                rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                valid = valid & (status.reshape(-1) == 1) & inside & on_background
        next_xy, valid = add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=step)
        next_pts = next_xy.reshape(-1, 1, 2)
        tracks.append({"frame": int(idx), "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records})
        prev_gray = gray
        prev_pts = next_pts
    return tracks

def render_surface_grid_frame(frame, track, prev_track=None):
    import cv2
    image = frame.copy()
    bg_mask = track["bg_mask"]
    active_mask = track["active_mask"]
    image[bg_mask] = (image[bg_mask] * 0.72 + np.array([35, 175, 105]) * 0.28).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 45, 35]) * 0.65).astype(np.uint8)
    pts = track["points"]
    valid = track["valid"]
    prev_pts = prev_track["points"] if prev_track is not None and len(prev_track["points"]) == len(pts) else None
    for i, (x, y) in enumerate(pts):
        if not valid[i]:
            continue
        x_i, y_i = int(round(x)), int(round(y))
        if prev_pts is not None:
            px, py = prev_pts[i]
            cv2.arrowedLine(image, (int(round(px)), int(round(py))), (x_i, y_i), (0, 230, 255), 1, tipLength=0.25)
        cv2.circle(image, (x_i, y_i), 3, (255, 235, 0), -1)
        cv2.circle(image, (x_i, y_i), 3, (0, 0, 0), 1)
    return image

def surface_grid_summary(tracks):
    rows = []
    base = np.empty((0, 2), dtype=np.float32)
    for track in tracks:
        pts = track["points"]
        if len(pts) > len(base):
            base = np.vstack([base, pts[len(base):]]) if len(base) else pts.copy()
        rows.append(surface_grid_summary_row(track, base))
    return pd.DataFrame(rows)

def surface_grid_summary_row(track, base_points):
    pts = track["points"]
    valid = track["valid"]
    disp = pts - base_points if len(pts) == len(base_points) else np.zeros_like(pts)
    speed = np.linalg.norm(disp, axis=1) if len(disp) else np.asarray([])
    return {
        "frame": int(track["frame"]),
        "agents": int(len(track["records"])),
        "grid_points_total": int(len(pts)),
        "grid_points_tracked": int(valid.sum()),
        "background_pixels": int(track["bg_mask"].sum()),
        "median_dx_from_start": float(np.median(disp[valid, 0])) if valid.any() else np.nan,
        "median_dy_from_start": float(np.median(disp[valid, 1])) if valid.any() else np.nan,
        "median_speed_from_start": float(np.median(speed[valid])) if valid.any() else np.nan,
    }

def show_surface_grid_tracking_full_video(video_path, title, video_name, grid_step=32, max_frames=None):
    import cv2
    from IPython.display import Video
    ensure_surface_grid_dependencies()
    input_path = Path(video_path)
    output_path = Path("generated_videos") / video_name
    output_path.parent.mkdir(exist_ok=True)
    cap = cv2.VideoCapture(str(input_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    ok, frame_bgr = cap.read()
    if not ok:
        cap.release()
        raise RuntimeError(f"No frames read from {input_path}")
    first = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    bg_mask, active_mask, records = surface_grid_background_mask(first)
    points0 = make_surface_grid_points(bg_mask, step=grid_step)
    track = {"frame": 0, "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
    base_points = points0.copy()
    rows = [surface_grid_summary_row(track, base_points)]
    rendered_preview = render_surface_grid_frame(first, track)
    writer = imageio.get_writer(output_path, fps=fps, macro_block_size=1)
    writer.append_data(rendered_preview)
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    prev_track = track
    frame_no = 1
    try:
        while max_frames is None or frame_no < max_frames:
            ok, frame_bgr = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            bg_mask, active_mask, records = surface_grid_background_mask(frame)
            if len(prev_pts) == 0:
                next_pts = prev_pts.copy()
                valid = np.zeros_like(valid)
            else:
                next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
                if next_pts is None or status is None:
                    valid = np.zeros_like(valid)
                    next_pts = prev_pts.copy()
                else:
                    next_xy = next_pts.reshape(-1, 2)
                    h, w = bg_mask.shape
                    inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                    on_background = np.zeros(len(next_xy), dtype=bool)
                    rounded = np.floor(next_xy[inside]).astype(int)
                    rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                    rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                    on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                    valid = valid & (status.reshape(-1) == 1) & inside & on_background
            next_xy, valid = add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=grid_step)
            if len(next_xy) > len(base_points):
                base_points = np.vstack([base_points, next_xy[len(base_points):]]) if len(base_points) else next_xy.copy()
            next_pts = next_xy.reshape(-1, 1, 2)
            track = {"frame": frame_no, "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
            writer.append_data(render_surface_grid_frame(frame, track, prev_track))
            rows.append(surface_grid_summary_row(track, base_points))
            prev_gray = gray
            prev_pts = next_pts
            prev_track = track
            frame_no += 1
            if frame_no % 100 == 0:
                print(f"processed {frame_no}/{total_frames or '?'} frames")
    finally:
        cap.release()
        writer.close()
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered_preview)
    plt.title(title)
    plt.axis("off")
    plt.show()
    data = pd.DataFrame(rows)
    data["video_path"] = str(output_path)
    print(f"video frames: 0:{frame_no} from {input_path}")
    display(Video(str(output_path), embed=False, html_attributes="controls muted loop"))
    display(data.head(12))
    return output_path, data

def show_surface_grid_tracking_experiment(frames_batch, frame_idx, title, video_name, clip_frames=None, seed=0, grid_step=32):
    ensure_surface_grid_dependencies()
    indices = np.arange(len(frames_batch), dtype=int) if clip_frames is None else clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    tracks = track_surface_grid_points(frames_batch, indices, step=grid_step)
    selected_pos = int(np.argmin(np.abs(indices - frame_idx))) if len(indices) else 0
    rendered = [render_surface_grid_frame(frames_batch[int(i)], track, tracks[pos - 1] if pos > 0 else None) for pos, (i, track) in enumerate(zip(indices, tracks))]
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered[selected_pos])
    plt.title(title)
    plt.axis("off")
    plt.show()
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    video_path = show_video(np.asarray(rendered), video_name, fps=4)
    data = surface_grid_summary(tracks)
    data["video_path"] = str(video_path)
    display(data.head(12))
    return tracks, np.asarray(rendered), data
ensure_surface_grid_dependencies()



### 1 - Lunar Lander Surface Grid Tracking


In [ ]:
bg_surface_grid_1_lunar_tracks, bg_surface_grid_1_lunar_frames, bg_surface_grid_1_lunar_data = show_surface_grid_tracking_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Background Surface Grid Tracking",
    video_name="surface_grid_1_lunar.mp4",
)


### 2 - Car Racing Surface Grid Tracking


In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import urllib.request
import cv2
import numpy as np

solved_car_path = PROJECT_ROOT / "external_videos/2_car_racing_solved_ppo_replay.mp4")
solved_car_path.parent.mkdir(parents=True, exist_ok=True)

solved_car_url = "https://huggingface.co/Brain33/ppo-car-racing-v3/resolve/main/replay.mp4"

if not solved_car_path.exists():
    urllib.request.urlretrieve(solved_car_url, solved_car_path)

cap = cv2.VideoCapture(str(solved_car_path))
frames_car = []

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames_car.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

cap.release()

frames_car = np.stack(frames_car).astype(np.uint8)
car_frame_idx = min(120, len(frames_car) - 1)

print("Solved CarRacing replay frames:", frames_car.shape)

bg_surface_grid_2_car_tracks, bg_surface_grid_2_car_frames, bg_surface_grid_2_car_data = show_surface_grid_tracking_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Background Surface Grid Tracking",
    video_name="surface_grid_2_car.mp4",
)

### 3 - Recorded Traffic With People Surface Grid Tracking


In [ ]:
bg_surface_grid_3_traffic_tracks, bg_surface_grid_3_traffic_frames, bg_surface_grid_3_traffic_data = show_surface_grid_tracking_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Background Surface Grid Tracking",
    video_name="surface_grid_3_traffic.mp4",
)


### 4 - Random YouTube Driving Scene Surface Grid Tracking


In [ ]:
bg_surface_grid_4_youtube_video_path, bg_surface_grid_4_youtube_data = show_surface_grid_tracking_full_video(
    PROJECT_ROOT / "external_videos/problem_4_youtube_random.mp4"),
    "4 - Random YouTube Driving Scene - Background Surface Grid Tracking",
    video_name="surface_grid_4_youtube.mp4",
)


### 5 - Hard Vehicle-Crowd Interaction Surface Grid Tracking


In [ ]:
bg_surface_grid_5_hard_video_path, bg_surface_grid_5_hard_data = show_surface_grid_tracking_full_video(
    PROJECT_ROOT / "external_videos/problem_5_youtube_hardest.mp4"),
    "5 - Hard Vehicle-Crowd Interaction - Background Surface Grid Tracking",
    video_name="surface_grid_5_hard.mp4",
)


## Save

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "1__segmentation_and_background__lunar_lander"

out_video = video_dir / f"{artifact_name}.mp4"
out_table = table_dir / f"{artifact_name}.csv"

imageio.mimsave(
    out_video,
    bg_surface_grid_1_lunar_frames.astype("uint8"),
    fps=4
)

bg_surface_grid_1_lunar_data.to_csv(
    out_table,
    index=False
)

print("saved video:", out_video)
print("saved table:", out_table)

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "2__segmentation_and_background__car_racing_old_before_rerun"

out_video = video_dir / f"{artifact_name}.mp4"
out_table = table_dir / f"{artifact_name}.csv"

imageio.mimsave(
    out_video,
    bg_surface_grid_2_car_frames.astype("uint8"),
    fps=4
)

bg_surface_grid_2_car_data.to_csv(
    out_table,
    index=False
)

print("saved video:", out_video)
print("saved table:", out_table)

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "3__segmentation_and_background__recorded_traffic_people"

out_video = video_dir / f"{artifact_name}.mp4"
out_table = table_dir / f"{artifact_name}.csv"

imageio.mimsave(
    out_video,
    bg_surface_grid_3_traffic_frames.astype("uint8"),
    fps=4
)

bg_surface_grid_3_traffic_data.to_csv(
    out_table,
    index=False
)

print("saved video:", out_video)
print("saved table:", out_table)

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import shutil

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "4__segmentation_and_background__3m_drive_suburbs"

shutil.copy2(
    Path(bg_surface_grid_4_youtube_video_path),
    video_dir / f"{artifact_name}.mp4"
)

bg_surface_grid_4_youtube_data.to_csv(
    table_dir / f"{artifact_name}.csv",
    index=False
)

print("saved video:", video_dir / f"{artifact_name}.mp4")
print("saved table:", table_dir / f"{artifact_name}.csv")

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import shutil

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "5__segmentation_and_background__3m_vehicle_crowd_hard"

shutil.copy2(
    Path(bg_surface_grid_5_hard_video_path),
    video_dir / f"{artifact_name}.mp4"
)

bg_surface_grid_5_hard_data.to_csv(
    table_dir / f"{artifact_name}.csv",
    index=False
)

print("saved video:", video_dir / f"{artifact_name}.mp4")
print("saved table:", table_dir / f"{artifact_name}.csv")

In [ ]:
stop

## Depth Anything V2 Background Scene Grid


In [ ]:
# Depth Anything V2 setup.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
# Uses Hugging Face Transformers. First run may download the model.
!pip install transformers accelerate imageio[ffmpeg] -q
# Depth-based background scene helpers.
# Active agents come from the winning segmentation: people + vehicles. Depth/grid is applied only to inverse-agent background.
def ensure_active_agent_depth_dependencies():
    global active_agent_seg_model
    if "active_agent_seg_model" not in globals():
        from ultralytics import YOLO
        active_agent_seg_model = YOLO("models/yolov8n-seg.pt")
    if "ACTIVE_AGENT_CLASS_NAMES" not in globals():
        globals()["ACTIVE_AGENT_CLASS_NAMES"] = {"person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"}
    if "mask_to_record" not in globals():
        def mask_to_record(mask, object_id):
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                return None
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            return {"object_id": object_id, "mask": mask, "box": (x1, y1, x2, y2), "area": int(mask.sum()), "centroid": (float(xs.mean()), float(ys.mean()))}
        globals()["mask_to_record"] = mask_to_record
    if "active_agent_records_from_yolo_seg" not in globals():
        def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
            result = model(frame, verbose=False, conf=conf_min)[0]
            records = []
            if result.masks is None:
                return records
            masks = result.masks.data.cpu().numpy().astype(bool)
            if masks.shape[1:] != frame.shape[:2]:
                import cv2
                masks = np.asarray([cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool) for mask in masks])
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()
            for mask, class_id, conf in zip(masks, classes, confs):
                class_name = result.names[int(class_id)]
                if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
                    continue
                record = mask_to_record(mask, len(records))
                if record is None:
                    continue
                record["class_id"] = int(class_id)
                record["class_name"] = class_name
                record["confidence"] = float(conf)
                records.append(record)
            return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
        globals()["active_agent_records_from_yolo_seg"] = active_agent_records_from_yolo_seg
def depth_background_masks(frame, conf_min=0.25):
    ensure_active_agent_depth_dependencies()
    records = active_agent_records_from_yolo_seg(frame, model=active_agent_seg_model, conf_min=conf_min)
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    background_mask = ~active_mask
    return background_mask, active_mask, records
def normalized_depth(depth):
    depth = np.asarray(depth, dtype=np.float32)
    valid = np.isfinite(depth)
    if not valid.any():
        return np.zeros_like(depth, dtype=np.float32)
    lo, hi = np.percentile(depth[valid], [2, 98])
    if hi <= lo:
        return np.zeros_like(depth, dtype=np.float32)
    return np.clip((depth - lo) / (hi - lo), 0, 1)
def depth_scene_grid_points(background_mask, depth, step=42, min_depth_quantile=5, max_depth_quantile=95):
    depth = np.asarray(depth, dtype=np.float32)
    valid_depth = depth[background_mask & np.isfinite(depth)]
    if len(valid_depth) == 0:
        return np.empty((0, 3), dtype=np.float32)
    z_lo, z_hi = np.percentile(valid_depth, [min_depth_quantile, max_depth_quantile])
    h, w = background_mask.shape
    pts = []
    for y in range(20, h - 20, step):
        for x in range(20, w - 20, step):
            if background_mask[y, x] and z_lo <= depth[y, x] <= z_hi:
                pts.append((x, y, float(depth[y, x])))
    return np.asarray(pts, dtype=np.float32)
def render_depth_background_scene(frame, depth, background_mask, active_mask, points, title):
    depth_norm = normalized_depth(depth)
    cmap = plt.get_cmap("turbo")
    depth_rgb = (cmap(depth_norm)[..., :3] * 255).astype(np.uint8)
    image = frame.copy()
    image[background_mask] = (image[background_mask] * 0.35 + depth_rgb[background_mask] * 0.65).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 50, 40]) * 0.65).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    if len(points):
        z = normalized_depth(points[:, 2])
        sizes = 10 + 22 * (1 - z)
        plt.scatter(points[:, 0], points[:, 1], s=sizes, c="yellow", edgecolors="black", linewidths=0.3)
    plt.title(title)
    plt.axis("off")
    plt.show()
def ensure_depth_video_helpers():
    if "show_video" in globals():
        return
    VIDEO_DIR = PROJECT_ROOT / "generated_videos"
    VIDEO_DIR.mkdir(exist_ok=True)
    def save_video(frames, path, fps=20):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        frames = frames.astype(np.uint8)
        try:
            imageio.mimsave(path, frames, fps=fps)
            return path
        except Exception as exc:
            fallback = path.with_suffix(".gif")
            print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
            print(f"Saving GIF fallback: {fallback}")
            imageio.mimsave(fallback, frames, duration=1 / fps)
            return fallback
    def show_video(frames, name, fps=20, embed=True):
        from IPython.display import Image as DisplayImage, Video
        path = save_video(frames, VIDEO_DIR / name, fps=fps)
        if path.suffix.lower() == ".gif":
            display(DisplayImage(filename=str(path)))
        else:
            display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
        return path
    globals()["show_video"] = show_video
def render_depth_background_scene_image(frame, depth, background_mask, active_mask, points):
    depth_norm = normalized_depth(depth)
    cmap = plt.get_cmap("turbo")
    depth_rgb = (cmap(depth_norm)[..., :3] * 255).astype(np.uint8)
    image = frame.copy()
    image[background_mask] = (image[background_mask] * 0.35 + depth_rgb[background_mask] * 0.65).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 50, 40]) * 0.65).astype(np.uint8)
    if len(points):
        for x, y, _ in points.astype(int):
            yy, xx = int(y), int(x)
            image[max(0, yy-2):yy+3, max(0, xx-2):xx+3] = np.array([255, 230, 0], dtype=np.uint8)
    return image
def save_depth_background_scene_video(frames_batch, name, depth_fn, model_name, fps=4, clip_frames=18, seed=0):
    ensure_depth_video_helpers()
    if "clip_indices" in globals():
        indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    else:
        indices = np.arange(min(len(frames_batch), clip_frames), dtype=int)
    rendered = []
    errors = []
    for i in indices:
        frame = frames_batch[i]
        background_mask, active_mask, records = depth_background_masks(frame)
        try:
            depth = depth_fn(frame)
            points = depth_scene_grid_points(background_mask, depth)
        except Exception as exc:
            depth = np.zeros(frame.shape[:2], dtype=np.float32)
            points = np.empty((0, 3), dtype=np.float32)
            errors.append(f"frame {int(i)}: {type(exc).__name__}: {exc}")
        rendered.append(render_depth_background_scene_image(frame, depth, background_mask, active_mask, points))
    if errors:
        print(f"{model_name} video fallback/errors:")
        print(errors[:3])
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    return show_video(np.asarray(rendered), name, fps=fps), errors
def show_depth_background_scene_experiment(frames_batch, frame_idx, title, depth_fn, model_name, video_name=None):
    frame = frames_batch[frame_idx]
    background_mask, active_mask, records = depth_background_masks(frame)
    try:
        depth = depth_fn(frame)
        points = depth_scene_grid_points(background_mask, depth)
        error = None
    except Exception as exc:
        depth = np.zeros(frame.shape[:2], dtype=np.float32)
        points = np.empty((0, 3), dtype=np.float32)
        error = f"{type(exc).__name__}: {exc}"
        print(f"{model_name} skipped: {error}")
    render_depth_background_scene(frame, depth, background_mask, active_mask, points, title)
    video_path = None
    if video_name is not None:
        video_path, video_errors = save_depth_background_scene_video(frames_batch, video_name, depth_fn=depth_fn, model_name=model_name)
        if error is None and video_errors:
            error = "; ".join(video_errors[:2])
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "model": model_name,
        "agents_removed": len(records),
        "background_pixels": int(background_mask.sum()),
        "grid_points": int(len(points)),
        "video_path": str(video_path) if video_path is not None else None,
        "error": error,
    }])
    display(summary)
    return depth, points, background_mask, summary
_DEPTH_ANYTHING_V2_PIPE = None
DEPTH_ANYTHING_V2_MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
def estimate_depth_anything_v2(frame):
    global _DEPTH_ANYTHING_V2_PIPE
    if _DEPTH_ANYTHING_V2_PIPE is None:
        from transformers import pipeline
        _DEPTH_ANYTHING_V2_PIPE = pipeline("depth-estimation", model=DEPTH_ANYTHING_V2_MODEL_ID, device=-1)
    result = _DEPTH_ANYTHING_V2_PIPE(Image.fromarray(frame.astype(np.uint8)))
    depth = np.asarray(result["depth"].resize((frame.shape[1], frame.shape[0])), dtype=np.float32)
    return depth


### 1 - Lunar Lander


In [ ]:
bg_depth_anything_v2_1_lunar_depth, bg_depth_anything_v2_1_lunar_points, bg_depth_anything_v2_1_lunar_mask, bg_depth_anything_v2_1_lunar_data = show_depth_background_scene_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Depth Anything V2 Background Scene Grid",
    depth_fn=estimate_depth_anything_v2,
    model_name="Depth Anything V2 Small",
    video_name="depth_anything_v2_1_lunar.mp4"
)


### 2 - Car Racing


In [ ]:
bg_depth_anything_v2_2_car_depth, bg_depth_anything_v2_2_car_points, bg_depth_anything_v2_2_car_mask, bg_depth_anything_v2_2_car_data = show_depth_background_scene_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Depth Anything V2 Background Scene Grid",
    depth_fn=estimate_depth_anything_v2,
    model_name="Depth Anything V2 Small",
    video_name="depth_anything_v2_2_car.mp4"
)


### 3 - Recorded Traffic With People


In [ ]:
bg_depth_anything_v2_3_traffic_depth, bg_depth_anything_v2_3_traffic_points, bg_depth_anything_v2_3_traffic_mask, bg_depth_anything_v2_3_traffic_data = show_depth_background_scene_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Depth Anything V2 Background Scene Grid",
    depth_fn=estimate_depth_anything_v2,
    model_name="Depth Anything V2 Small",
    video_name="depth_anything_v2_3_traffic.mp4"
)


### 4 - Random YouTube Driving Scene


In [ ]:
bg_depth_anything_v2_4_youtube_depth, bg_depth_anything_v2_4_youtube_points, bg_depth_anything_v2_4_youtube_mask, bg_depth_anything_v2_4_youtube_data = show_depth_background_scene_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Depth Anything V2 Background Scene Grid",
    depth_fn=estimate_depth_anything_v2,
    model_name="Depth Anything V2 Small",
    video_name="depth_anything_v2_4_youtube.mp4"
)


### 5 - Hard Vehicle-Crowd Interaction


In [ ]:
bg_depth_anything_v2_5_hard_depth, bg_depth_anything_v2_5_hard_points, bg_depth_anything_v2_5_hard_mask, bg_depth_anything_v2_5_hard_data = show_depth_background_scene_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Depth Anything V2 Background Scene Grid",
    depth_fn=estimate_depth_anything_v2,
    model_name="Depth Anything V2 Small",
    video_name="depth_anything_v2_5_hard.mp4"
)


## Metric3D Background Scene Grid


In [ ]:
# Metric3D setup.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
# Uses PyTorch Hub: yvanyin/metric3d metric3d_vit_small. First run may download repo/checkpoint.
!pip install imageio[ffmpeg] -q
# Depth-based background scene helpers.
# Active agents come from the winning segmentation: people + vehicles. Depth/grid is applied only to inverse-agent background.
def ensure_active_agent_depth_dependencies():
    global active_agent_seg_model
    if "active_agent_seg_model" not in globals():
        from ultralytics import YOLO
        active_agent_seg_model = YOLO("models/yolov8n-seg.pt")
    if "ACTIVE_AGENT_CLASS_NAMES" not in globals():
        globals()["ACTIVE_AGENT_CLASS_NAMES"] = {"person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"}
    if "mask_to_record" not in globals():
        def mask_to_record(mask, object_id):
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                return None
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            return {"object_id": object_id, "mask": mask, "box": (x1, y1, x2, y2), "area": int(mask.sum()), "centroid": (float(xs.mean()), float(ys.mean()))}
        globals()["mask_to_record"] = mask_to_record
    if "active_agent_records_from_yolo_seg" not in globals():
        def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
            result = model(frame, verbose=False, conf=conf_min)[0]
            records = []
            if result.masks is None:
                return records
            masks = result.masks.data.cpu().numpy().astype(bool)
            if masks.shape[1:] != frame.shape[:2]:
                import cv2
                masks = np.asarray([cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool) for mask in masks])
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()
            for mask, class_id, conf in zip(masks, classes, confs):
                class_name = result.names[int(class_id)]
                if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
                    continue
                record = mask_to_record(mask, len(records))
                if record is None:
                    continue
                record["class_id"] = int(class_id)
                record["class_name"] = class_name
                record["confidence"] = float(conf)
                records.append(record)
            return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
        globals()["active_agent_records_from_yolo_seg"] = active_agent_records_from_yolo_seg
def depth_background_masks(frame, conf_min=0.25):
    ensure_active_agent_depth_dependencies()
    records = active_agent_records_from_yolo_seg(frame, model=active_agent_seg_model, conf_min=conf_min)
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    background_mask = ~active_mask
    return background_mask, active_mask, records
def normalized_depth(depth):
    depth = np.asarray(depth, dtype=np.float32)
    valid = np.isfinite(depth)
    if not valid.any():
        return np.zeros_like(depth, dtype=np.float32)
    lo, hi = np.percentile(depth[valid], [2, 98])
    if hi <= lo:
        return np.zeros_like(depth, dtype=np.float32)
    return np.clip((depth - lo) / (hi - lo), 0, 1)
def depth_scene_grid_points(background_mask, depth, step=42, min_depth_quantile=5, max_depth_quantile=95):
    depth = np.asarray(depth, dtype=np.float32)
    valid_depth = depth[background_mask & np.isfinite(depth)]
    if len(valid_depth) == 0:
        return np.empty((0, 3), dtype=np.float32)
    z_lo, z_hi = np.percentile(valid_depth, [min_depth_quantile, max_depth_quantile])
    h, w = background_mask.shape
    pts = []
    for y in range(20, h - 20, step):
        for x in range(20, w - 20, step):
            if background_mask[y, x] and z_lo <= depth[y, x] <= z_hi:
                pts.append((x, y, float(depth[y, x])))
    return np.asarray(pts, dtype=np.float32)
def render_depth_background_scene(frame, depth, background_mask, active_mask, points, title):
    depth_norm = normalized_depth(depth)
    cmap = plt.get_cmap("turbo")
    depth_rgb = (cmap(depth_norm)[..., :3] * 255).astype(np.uint8)
    image = frame.copy()
    image[background_mask] = (image[background_mask] * 0.35 + depth_rgb[background_mask] * 0.65).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 50, 40]) * 0.65).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    if len(points):
        z = normalized_depth(points[:, 2])
        sizes = 10 + 22 * (1 - z)
        plt.scatter(points[:, 0], points[:, 1], s=sizes, c="yellow", edgecolors="black", linewidths=0.3)
    plt.title(title)
    plt.axis("off")
    plt.show()
def ensure_depth_video_helpers():
    if "show_video" in globals():
        return
    VIDEO_DIR = PROJECT_ROOT / "generated_videos"
    VIDEO_DIR.mkdir(exist_ok=True)
    def save_video(frames, path, fps=20):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        frames = frames.astype(np.uint8)
        try:
            imageio.mimsave(path, frames, fps=fps)
            return path
        except Exception as exc:
            fallback = path.with_suffix(".gif")
            print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
            print(f"Saving GIF fallback: {fallback}")
            imageio.mimsave(fallback, frames, duration=1 / fps)
            return fallback
    def show_video(frames, name, fps=20, embed=True):
        from IPython.display import Image as DisplayImage, Video
        path = save_video(frames, VIDEO_DIR / name, fps=fps)
        if path.suffix.lower() == ".gif":
            display(DisplayImage(filename=str(path)))
        else:
            display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
        return path
    globals()["show_video"] = show_video
def render_depth_background_scene_image(frame, depth, background_mask, active_mask, points):
    depth_norm = normalized_depth(depth)
    cmap = plt.get_cmap("turbo")
    depth_rgb = (cmap(depth_norm)[..., :3] * 255).astype(np.uint8)
    image = frame.copy()
    image[background_mask] = (image[background_mask] * 0.35 + depth_rgb[background_mask] * 0.65).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 50, 40]) * 0.65).astype(np.uint8)
    if len(points):
        for x, y, _ in points.astype(int):
            yy, xx = int(y), int(x)
            image[max(0, yy-2):yy+3, max(0, xx-2):xx+3] = np.array([255, 230, 0], dtype=np.uint8)
    return image
def save_depth_background_scene_video(frames_batch, name, depth_fn, model_name, fps=4, clip_frames=18, seed=0):
    ensure_depth_video_helpers()
    if "clip_indices" in globals():
        indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    else:
        indices = np.arange(min(len(frames_batch), clip_frames), dtype=int)
    rendered = []
    errors = []
    for i in indices:
        frame = frames_batch[i]
        background_mask, active_mask, records = depth_background_masks(frame)
        try:
            depth = depth_fn(frame)
            points = depth_scene_grid_points(background_mask, depth)
        except Exception as exc:
            depth = np.zeros(frame.shape[:2], dtype=np.float32)
            points = np.empty((0, 3), dtype=np.float32)
            errors.append(f"frame {int(i)}: {type(exc).__name__}: {exc}")
        rendered.append(render_depth_background_scene_image(frame, depth, background_mask, active_mask, points))
    if errors:
        print(f"{model_name} video fallback/errors:")
        print(errors[:3])
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    return show_video(np.asarray(rendered), name, fps=fps), errors
def show_depth_background_scene_experiment(frames_batch, frame_idx, title, depth_fn, model_name, video_name=None):
    frame = frames_batch[frame_idx]
    background_mask, active_mask, records = depth_background_masks(frame)
    try:
        depth = depth_fn(frame)
        points = depth_scene_grid_points(background_mask, depth)
        error = None
    except Exception as exc:
        depth = np.zeros(frame.shape[:2], dtype=np.float32)
        points = np.empty((0, 3), dtype=np.float32)
        error = f"{type(exc).__name__}: {exc}"
        print(f"{model_name} skipped: {error}")
    render_depth_background_scene(frame, depth, background_mask, active_mask, points, title)
    video_path = None
    if video_name is not None:
        video_path, video_errors = save_depth_background_scene_video(frames_batch, video_name, depth_fn=depth_fn, model_name=model_name)
        if error is None and video_errors:
            error = "; ".join(video_errors[:2])
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "model": model_name,
        "agents_removed": len(records),
        "background_pixels": int(background_mask.sum()),
        "grid_points": int(len(points)),
        "video_path": str(video_path) if video_path is not None else None,
        "error": error,
    }])
    display(summary)
    return depth, points, background_mask, summary
_METRIC3D_MODEL = None
def estimate_metric3d_depth(frame):
    global _METRIC3D_MODEL
    import torch
    import cv2
    if _METRIC3D_MODEL is None:
        try:
            _METRIC3D_MODEL = torch.hub.load("yvanyin/metric3d", "metric3d_vit_small", pretrain=True, trust_repo=True)
        except TypeError:
            _METRIC3D_MODEL = torch.hub.load("yvanyin/metric3d", "metric3d_vit_small", pretrain=True)
        _METRIC3D_MODEL.eval()
    input_size = (616, 1064)
    h, w = frame.shape[:2]
    scale = min(input_size[0] / h, input_size[1] / w)
    resized = cv2.resize(frame, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LINEAR)
    rh, rw = resized.shape[:2]
    pad_h, pad_w = input_size[0] - rh, input_size[1] - rw
    pad_top, pad_left = pad_h // 2, pad_w // 2
    padded = cv2.copyMakeBorder(resized, pad_top, pad_h - pad_top, pad_left, pad_w - pad_left, cv2.BORDER_CONSTANT, value=(123.675, 116.28, 103.53))
    mean = torch.tensor([123.675, 116.28, 103.53]).float()[:, None, None]
    std = torch.tensor([58.395, 57.12, 57.375]).float()[:, None, None]
    rgb = torch.from_numpy(padded.transpose(2, 0, 1)).float()
    rgb = ((rgb - mean) / std)[None]
    with torch.no_grad():
        pred_depth, confidence, output_dict = _METRIC3D_MODEL.inference({"input": rgb})
    pred = pred_depth.squeeze().detach().cpu()
    pred = pred[pad_top:pad_top + rh, pad_left:pad_left + rw]
    pred = torch.nn.functional.interpolate(pred[None, None], size=(h, w), mode="bilinear", align_corners=False).squeeze()
    return pred.numpy().astype(np.float32)


### 1 - Lunar Lander


In [ ]:
bg_metric3d_1_lunar_depth, bg_metric3d_1_lunar_points, bg_metric3d_1_lunar_mask, bg_metric3d_1_lunar_data = show_depth_background_scene_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Metric3D Background Scene Grid",
    depth_fn=estimate_metric3d_depth,
    model_name="Metric3D ViT Small",
    video_name="metric3d_1_lunar.mp4"
)


### 2 - Car Racing


In [ ]:
bg_metric3d_2_car_depth, bg_metric3d_2_car_points, bg_metric3d_2_car_mask, bg_metric3d_2_car_data = show_depth_background_scene_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Metric3D Background Scene Grid",
    depth_fn=estimate_metric3d_depth,
    model_name="Metric3D ViT Small",
    video_name="metric3d_2_car.mp4"
)


### 3 - Recorded Traffic With People


In [ ]:
bg_metric3d_3_traffic_depth, bg_metric3d_3_traffic_points, bg_metric3d_3_traffic_mask, bg_metric3d_3_traffic_data = show_depth_background_scene_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Metric3D Background Scene Grid",
    depth_fn=estimate_metric3d_depth,
    model_name="Metric3D ViT Small",
    video_name="metric3d_3_traffic.mp4"
)


### 4 - Random YouTube Driving Scene


In [ ]:
bg_metric3d_4_youtube_depth, bg_metric3d_4_youtube_points, bg_metric3d_4_youtube_mask, bg_metric3d_4_youtube_data = show_depth_background_scene_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Metric3D Background Scene Grid",
    depth_fn=estimate_metric3d_depth,
    model_name="Metric3D ViT Small",
    video_name="metric3d_4_youtube.mp4"
)


### 5 - Hard Vehicle-Crowd Interaction


In [ ]:
bg_metric3d_5_hard_depth, bg_metric3d_5_hard_points, bg_metric3d_5_hard_mask, bg_metric3d_5_hard_data = show_depth_background_scene_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Metric3D Background Scene Grid",
    depth_fn=estimate_metric3d_depth,
    model_name="Metric3D ViT Small",
    video_name="metric3d_5_hard.mp4"
)


## VGGT 3D Background Scene Reconstruction


In [ ]:
# VGGT setup: multi-frame 3D scene reconstruction candidate.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import tempfile
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
# Uses official VGGT package/model. First real run can download code/model weights.
!pip install "git+https://github.com/facebookresearch/vggt.git" "imageio[ffmpeg]" -q
# VGGT predicts camera/depth/point maps/tracks from frames. We mask active agents first, using the winning people+vehicles segmentation.
VGGT_RUN_REAL_MODEL = False  # keep examples fast; set True only when you want to download/run VGGT-1B
def ensure_vggt_scene_dependencies():
    global active_agent_seg_model
    if "active_agent_seg_model" not in globals():
        from ultralytics import YOLO
        active_agent_seg_model = YOLO("models/yolov8n-seg.pt")
    if "ACTIVE_AGENT_CLASS_NAMES" not in globals():
        globals()["ACTIVE_AGENT_CLASS_NAMES"] = {"person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"}
    if "mask_to_record" not in globals():
        def mask_to_record(mask, object_id):
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                return None
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            return {"object_id": object_id, "mask": mask, "box": (x1, y1, x2, y2), "area": int(mask.sum()), "centroid": (float(xs.mean()), float(ys.mean()))}
        globals()["mask_to_record"] = mask_to_record
    if "active_agent_records_from_yolo_seg" not in globals():
        def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
            result = model(frame, verbose=False, conf=conf_min)[0]
            records = []
            if result.masks is None:
                return records
            masks = result.masks.data.cpu().numpy().astype(bool)
            if masks.shape[1:] != frame.shape[:2]:
                import cv2
                masks = np.asarray([cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool) for mask in masks])
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()
            for mask, class_id, conf in zip(masks, classes, confs):
                class_name = result.names[int(class_id)]
                if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
                    continue
                record = mask_to_record(mask, len(records))
                if record is None:
                    continue
                record["class_id"] = int(class_id)
                record["class_name"] = class_name
                record["confidence"] = float(conf)
                records.append(record)
            return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
        globals()["active_agent_records_from_yolo_seg"] = active_agent_records_from_yolo_seg
    if "clip_indices" not in globals():
        def clip_indices(num_frames, clip_frames=48, seed=0):
            if num_frames <= 0:
                return np.asarray([], dtype=int)
            clip_frames = min(int(clip_frames), int(num_frames))
            rng = np.random.default_rng(seed)
            max_start = max(0, int(num_frames) - clip_frames)
            start = int(rng.integers(0, max_start + 1)) if max_start else 0
            return np.arange(start, start + clip_frames, dtype=int)
        globals()["clip_indices"] = clip_indices
    if "show_video" not in globals():
        VIDEO_DIR = PROJECT_ROOT / "generated_videos"
        VIDEO_DIR.mkdir(exist_ok=True)
        def save_video(frames, path, fps=20):
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            frames = frames.astype(np.uint8)
            try:
                imageio.mimsave(path, frames, fps=fps)
                return path
            except Exception as exc:
                fallback = path.with_suffix(".gif")
                print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
                print(f"Saving GIF fallback: {fallback}")
                imageio.mimsave(fallback, frames, duration=1 / fps)
                return fallback
        def show_video(frames, name, fps=20, embed=True):
            from IPython.display import Image as DisplayImage, Video
            path = save_video(frames, VIDEO_DIR / name, fps=fps)
            if path.suffix.lower() == ".gif":
                display(DisplayImage(filename=str(path)))
            else:
                display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
            return path
        globals()["show_video"] = show_video
    if "read_video" not in globals():
        def read_video(path, max_frames=None, stride=1, start_frame=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            if start_frame:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
            frames_out = []
            frame_no = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_no % stride == 0:
                    frames_out.append(frame[:, :, ::-1])
                    if max_frames is not None and len(frames_out) >= max_frames:
                        break
                frame_no += 1
            cap.release()
            if not frames_out:
                raise RuntimeError(f"No frames read from {path}")
            return np.stack(frames_out)
        globals()["read_video"] = read_video
    if "read_video_sample" not in globals():
        def read_video_sample(path, max_frames=120, stride=3, start_seconds=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            cap.release()
            return read_video(path, max_frames=max_frames, stride=stride, start_frame=int(start_seconds * fps))
        globals()["read_video_sample"] = read_video_sample
    if "frames" not in globals():
        try:
            import gymnasium as gym
            from gymnasium.envs.box2d.lunar_lander import heuristic
            env = gym.make("LunarLander-v3", render_mode="rgb_array")
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(80):
                action = int(heuristic(env.unwrapped, obs))
                obs, _, terminated, truncated, _ = env.step(action)
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames"] = np.stack(out)
            globals()["frame_idx"] = min(10, len(frames) - 1)
        except Exception as exc:
            print(f"Lunar standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_car" not in globals():
        try:
            import gymnasium as gym
            env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(120):
                obs, _, terminated, truncated, _ = env.step(env.action_space.sample())
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames_car"] = np.stack(out)
            globals()["car_frame_idx"] = min(40, len(frames_car) - 1)
        except Exception as exc:
            print(f"Car standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_medium" not in globals() and PROJECT_ROOT / "external_videos/traffic.avi").exists():
        globals()["frames_medium"] = read_video("external_videos/traffic.avi", max_frames=100, stride=1)
        globals()["traffic_frame_idx"] = min(1, len(frames_medium) - 1)
    if "frames_problem4" not in globals() and PROJECT_ROOT / "external_videos/problem_4_youtube_random.mp4").exists():
        globals()["frames_problem4"] = read_video_sample("external_videos/problem_4_youtube_random.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem4_frame_idx"] = min(30, len(frames_problem4) - 1)
    if "frames_problem5" not in globals() and PROJECT_ROOT / "external_videos/problem_5_youtube_hardest.mp4").exists():
        globals()["frames_problem5"] = read_video_sample("external_videos/problem_5_youtube_hardest.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem5_frame_idx"] = min(40, len(frames_problem5) - 1)
def vggt_agent_mask(frame, conf_min=0.25):
    ensure_vggt_scene_dependencies()
    records = active_agent_records_from_yolo_seg(frame, model=active_agent_seg_model, conf_min=conf_min)
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    return active_mask, ~active_mask, records
def vggt_masked_frames(frames_batch, indices):
    masked = []
    masks = []
    records_per_frame = []
    for i in indices:
        frame = frames_batch[int(i)]
        active_mask, background_mask, records = vggt_agent_mask(frame)
        image = frame.copy()
        if active_mask.any():
            fill = np.median(image[background_mask], axis=0).astype(np.uint8) if background_mask.any() else np.array([0, 0, 0], dtype=np.uint8)
            image[active_mask] = fill
        masked.append(image)
        masks.append(background_mask)
        records_per_frame.append(records)
    return np.asarray(masked), masks, records_per_frame
_VGGT_MODEL = None
def load_vggt_model():
    global _VGGT_MODEL
    if _VGGT_MODEL is None:
        import torch
        from vggt.models.vggt import VGGT
        device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        _VGGT_MODEL = VGGT.from_pretrained("facebook/VGGT-1B").to(device).eval()
        _VGGT_MODEL.device_name = device
    return _VGGT_MODEL
def run_vggt_scene_reconstruction(masked_frames, max_frames=8):
    import torch
    from vggt.utils.load_fn import load_and_preprocess_images
    model = load_vggt_model()
    device = getattr(model, "device_name", next(model.parameters()).device.type)
    selected = masked_frames[:max_frames]
    with tempfile.TemporaryDirectory() as tmp:
        image_paths = []
        for i, frame in enumerate(selected):
            path = Path(tmp) / f"frame_{i:03d}.png"
            Image.fromarray(frame.astype(np.uint8)).save(path)
            image_paths.append(str(path))
        images = load_and_preprocess_images(image_paths).to(device)
        if images.ndim == 4:
            images = images[None]
        dtype = torch.float16 if device in {"cuda", "mps"} else torch.float32
        with torch.no_grad():
            if device == "cuda":
                with torch.cuda.amp.autocast(dtype=dtype):
                    predictions = model(images)
            else:
                predictions = model(images)
    depth = None
    point_map = None
    if isinstance(predictions, dict):
        for key in ["depth", "depth_map", "depth_maps"]:
            if key in predictions:
                depth = predictions[key]
                break
        for key in ["point_map", "point_maps", "world_points", "points3d"]:
            if key in predictions:
                point_map = predictions[key]
                break
    if depth is not None:
        depth_np = np.squeeze(depth.detach().float().cpu().numpy())
    elif point_map is not None:
        pts = np.squeeze(point_map.detach().float().cpu().numpy())
        if pts.ndim == 4:
            depth_np = pts[..., 2]
        elif pts.ndim == 3 and pts.shape[-1] == 3:
            depth_np = pts[..., 2][None]
        else:
            raise RuntimeError(f"Unexpected VGGT point map shape: {pts.shape}")
    else:
        raise RuntimeError(f"VGGT predictions did not expose depth/point map keys: {list(predictions.keys()) if isinstance(predictions, dict) else type(predictions)}")
    if depth_np.ndim == 2:
        depth_np = depth_np[None]
    if depth_np.ndim > 3:
        depth_np = np.reshape(depth_np, (-1,) + depth_np.shape[-2:])
    resized = []
    for depth_frame in depth_np[:len(masked_frames)]:
        depth_image = Image.fromarray(depth_frame.astype(np.float32))
        resized.append(np.asarray(depth_image.resize((masked_frames[0].shape[1], masked_frames[0].shape[0]))).astype(np.float32))
    return np.asarray(resized), predictions
def vggt_fallback_depth(frame):
    h, w = frame.shape[:2]
    yy, xx = np.indices((h, w))
    return (0.85 * yy + 0.15 * np.abs(xx - w / 2)).astype(np.float32)
def vggt_normalized_depth(depth):
    depth = np.asarray(depth, dtype=np.float32)
    valid = np.isfinite(depth)
    if not valid.any():
        return np.zeros_like(depth, dtype=np.float32)
    lo, hi = np.percentile(depth[valid], [2, 98])
    if hi <= lo:
        return np.zeros_like(depth, dtype=np.float32)
    return np.clip((depth - lo) / (hi - lo), 0, 1)
def vggt_scene_points(background_mask, depth, step=42):
    valid = background_mask & np.isfinite(depth)
    if not valid.any():
        return np.empty((0, 3), dtype=np.float32)
    z_lo, z_hi = np.percentile(depth[valid], [5, 95])
    h, w = background_mask.shape
    pts = []
    for y in range(20, h - 20, step):
        for x in range(20, w - 20, step):
            if background_mask[y, x] and z_lo <= depth[y, x] <= z_hi:
                pts.append((x, y, float(depth[y, x])))
    return np.asarray(pts, dtype=np.float32)
def render_vggt_scene_image(frame, depth, background_mask, active_mask, points):
    depth_norm = vggt_normalized_depth(depth)
    depth_rgb = (plt.get_cmap("turbo")(depth_norm)[..., :3] * 255).astype(np.uint8)
    image = frame.copy()
    image[background_mask] = (image[background_mask] * 0.3 + depth_rgb[background_mask] * 0.7).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.25 + np.array([230, 45, 35]) * 0.75).astype(np.uint8)
    for x, y, z in points.astype(int):
        image[max(0, y-2):y+3, max(0, x-2):x+3] = np.array([255, 230, 0], dtype=np.uint8)
    return image
def show_vggt_scene_reconstruction_experiment(frames_batch, frame_idx, title, video_name=None, clip_frames=8, seed=0, run_model=None):
    ensure_vggt_scene_dependencies()
    if run_model is None:
        run_model = VGGT_RUN_REAL_MODEL
    indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    if len(indices) == 0:
        raise RuntimeError("No frames for VGGT scene reconstruction")
    masked_frames, bg_masks, records_per_frame = vggt_masked_frames(frames_batch, indices)
    selected_pos = int(np.argmin(np.abs(indices - frame_idx)))
    frame = frames_batch[int(indices[selected_pos])]
    active_mask = ~bg_masks[selected_pos]
    error = None
    prediction_keys = []
    try:
        if not run_model:
            raise RuntimeError("VGGT real model disabled for plumbing test")
        depth_sequence, predictions = run_vggt_scene_reconstruction(masked_frames, max_frames=min(clip_frames, 8))
        depth = depth_sequence[selected_pos] if selected_pos < len(depth_sequence) else depth_sequence[0]
        if isinstance(predictions, dict):
            prediction_keys = list(predictions.keys())
        model_status = "VGGT real inference"
    except Exception as exc:
        depth = vggt_fallback_depth(frame)
        error = f"{type(exc).__name__}: {exc}"
        model_status = "fallback depth overlay; VGGT did not run"
        print(f"VGGT skipped: {error}")
    points = vggt_scene_points(bg_masks[selected_pos], depth)
    rendered = render_vggt_scene_image(frame, depth, bg_masks[selected_pos], active_mask, points)
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered)
    plt.title(title)
    plt.axis("off")
    plt.show()
    video_path = None
    if video_name is not None:
        video_frames = []
        for pos, i in enumerate(indices):
            f = frames_batch[int(i)]
            if error:
                d = depth if pos == selected_pos else vggt_fallback_depth(f)
            else:
                d = depth_sequence[pos] if pos < len(depth_sequence) else depth_sequence[-1]
            if d.shape != f.shape[:2]:
                d = np.asarray(Image.fromarray(d.astype(np.float32)).resize((f.shape[1], f.shape[0]))).astype(np.float32)
            pts = vggt_scene_points(bg_masks[pos], d)
            video_frames.append(render_vggt_scene_image(f, d, bg_masks[pos], ~bg_masks[pos], pts))
        print(f"video frames: {int(indices[0])}:{int(indices[-1]) + 1}")
        video_path = show_video(np.asarray(video_frames), video_name, fps=4)
    summary = pd.DataFrame([{
        "frame": int(indices[selected_pos]),
        "model": "VGGT-1B",
        "status": model_status,
        "input_frames": int(len(indices)),
        "agents_removed": int(len(records_per_frame[selected_pos])),
        "background_pixels": int(bg_masks[selected_pos].sum()),
        "scene_points": int(len(points)),
        "prediction_keys": ", ".join(prediction_keys[:8]),
        "video_path": str(video_path) if video_path is not None else None,
        "error": error,
    }])
    display(summary)
    return depth, points, bg_masks[selected_pos], summary
ensure_vggt_scene_dependencies()



### 1 - Lunar Lander


In [ ]:
bg_vggt_1_lunar_depth, bg_vggt_1_lunar_points, bg_vggt_1_lunar_mask, bg_vggt_1_lunar_data = show_vggt_scene_reconstruction_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - VGGT 3D Background Scene Reconstruction",
    video_name="vggt_1_lunar.mp4"
)


### 2 - Car Racing


In [ ]:
bg_vggt_2_car_depth, bg_vggt_2_car_points, bg_vggt_2_car_mask, bg_vggt_2_car_data = show_vggt_scene_reconstruction_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - VGGT 3D Background Scene Reconstruction",
    video_name="vggt_2_car.mp4"
)


### 3 - Recorded Traffic With People


In [ ]:
bg_vggt_3_traffic_depth, bg_vggt_3_traffic_points, bg_vggt_3_traffic_mask, bg_vggt_3_traffic_data = show_vggt_scene_reconstruction_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - VGGT 3D Background Scene Reconstruction",
    video_name="vggt_3_traffic.mp4"
)


### 4 - Random YouTube Driving Scene


In [ ]:
bg_vggt_4_youtube_depth, bg_vggt_4_youtube_points, bg_vggt_4_youtube_mask, bg_vggt_4_youtube_data = show_vggt_scene_reconstruction_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - VGGT 3D Background Scene Reconstruction",
    video_name="vggt_4_youtube.mp4"
)


### 5 - Hard Vehicle-Crowd Interaction


In [ ]:
bg_vggt_5_hard_depth, bg_vggt_5_hard_points, bg_vggt_5_hard_mask, bg_vggt_5_hard_data = show_vggt_scene_reconstruction_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - VGGT 3D Background Scene Reconstruction",
    video_name="vggt_5_hard.mp4"
)


## Lucas-Kanade Optical Flow


In [ ]:
# Dynamic background segmentation via Lucas-Kanade point tracking
# Background is represented as tracked reference points, not as a static mask.
import pandas as pd
import cv2
def gray_frame(frame):
    return cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
def track_background_points(frame_t, frame_t1, max_corners=250, quality_level=0.01, min_distance=8):
    gray_t = gray_frame(frame_t)
    gray_t1 = gray_frame(frame_t1)
    points_t = cv2.goodFeaturesToTrack(
        gray_t,
        maxCorners=max_corners,
        qualityLevel=quality_level,
        minDistance=min_distance,
        blockSize=7,
    )
    if points_t is None:
        return pd.DataFrame(columns=["point_id", "x_t", "y_t", "x_t1", "y_t1", "dx", "dy", "speed", "is_background"])
    points_t1, status, error = cv2.calcOpticalFlowPyrLK(
        gray_t,
        gray_t1,
        points_t,
        None,
        winSize=(21, 21),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
    )
    status = status.reshape(-1).astype(bool)
    valid_t = points_t.reshape(-1, 2)[status]
    valid_t1 = points_t1.reshape(-1, 2)[status]
    flow = valid_t1 - valid_t
    speed = np.linalg.norm(flow, axis=1)
    if len(flow) == 0:
        return pd.DataFrame(columns=["point_id", "x_t", "y_t", "x_t1", "y_t1", "dx", "dy", "speed", "is_background"])
    median_flow = np.median(flow, axis=0)
    residual = np.linalg.norm(flow - median_flow, axis=1)
    threshold = np.median(residual) + 2.5 * np.median(np.abs(residual - np.median(residual)))
    if threshold <= 1e-6:
        threshold = np.percentile(residual, 75) + 1e-6
    is_background = residual <= threshold
    return pd.DataFrame({
        "point_id": np.arange(len(valid_t)),
        "x_t": valid_t[:, 0],
        "y_t": valid_t[:, 1],
        "x_t1": valid_t1[:, 0],
        "y_t1": valid_t1[:, 1],
        "dx": flow[:, 0],
        "dy": flow[:, 1],
        "speed": speed,
        "residual_from_dominant_motion": residual,
        "is_background": is_background,
    })
def background_motion_summary(track_df):
    bg = track_df[track_df["is_background"]]
    if len(bg) == 0:
        bg = track_df
    if len(bg) == 0:
        return {"bg_dx": 0.0, "bg_dy": 0.0, "bg_speed": 0.0, "points": 0}
    bg_dx = float(bg["dx"].median())
    bg_dy = float(bg["dy"].median())
    return {
        "bg_dx": bg_dx,
        "bg_dy": bg_dy,
        "bg_speed": float(np.hypot(bg_dx, bg_dy)),
        "points": int(len(bg)),
    }
def show_background_tracking(frame_t, track_df, title, max_arrows=120):
    bg = track_df[track_df["is_background"]].copy()
    fg = track_df[~track_df["is_background"]].copy()
    if len(bg) > max_arrows:
        bg = bg.sample(max_arrows, random_state=0)
    if len(fg) > max_arrows // 3:
        fg = fg.sample(max_arrows // 3, random_state=1)
    summary = background_motion_summary(track_df)
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(frame_t)
    if len(bg):
        ax.quiver(bg["x_t"], bg["y_t"], bg["dx"], bg["dy"], color="lime", angles="xy", scale_units="xy", scale=1, width=0.003)
    if len(fg):
        ax.scatter(fg["x_t"], fg["y_t"], s=12, c="red", alpha=0.8, label="outlier motion")
    h, w = frame_t.shape[:2]
    ax.quiver([w * 0.08], [h * 0.1], [summary["bg_dx"] * 5], [summary["bg_dy"] * 5], color="yellow", angles="xy", scale_units="xy", scale=1, width=0.008)
    ax.text(w * 0.08, h * 0.1 + 18, f"median bg: dx={summary['bg_dx']:.2f}, dy={summary['bg_dy']:.2f}", color="yellow", weight="bold")
    ax.set_title(title)
    ax.axis("off")
    plt.show()
def plot_background_motion(track_df, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    colors = np.where(track_df["is_background"], "tab:green", "tab:red")
    axes[0].scatter(track_df["dx"], track_df["dy"], s=10, c=colors, alpha=0.75)
    axes[0].axvline(0, color="black", linewidth=0.8, alpha=0.4)
    axes[0].axhline(0, color="black", linewidth=0.8, alpha=0.4)
    axes[0].set_title("point motion vectors")
    axes[0].set_xlabel("dx")
    axes[0].set_ylabel("dy")
    axes[0].grid(alpha=0.25)
    axes[1].hist(track_df["speed"], bins=30)
    axes[1].set_title("point speed distribution")
    axes[1].set_xlabel("speed")
    axes[1].set_ylabel("points")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
def show_background_data(track_df, title):
    display_cols = ["point_id", "x_t", "y_t", "x_t1", "y_t1", "dx", "dy", "speed", "residual_from_dominant_motion", "is_background"]
    rounded = track_df[display_cols].copy()
    for col in ["x_t", "y_t", "x_t1", "y_t1", "dx", "dy", "speed", "residual_from_dominant_motion"]:
        rounded[col] = rounded[col].round(2)
    print(title)
    print(background_motion_summary(track_df))
    display(rounded.head(25))
    return rounded
def render_background_tracking_frame(frame_t, track_df, max_arrows=120):
    import cv2
    image = frame_t.copy()
    bg = track_df[track_df["is_background"]].copy()
    fg = track_df[~track_df["is_background"]].copy()
    if len(bg) > max_arrows:
        bg = bg.sample(max_arrows, random_state=0)
    if len(fg) > max_arrows // 3:
        fg = fg.sample(max_arrows // 3, random_state=1)
    for _, row in bg.iterrows():
        p0 = (int(row["x_t"]), int(row["y_t"]))
        p1 = (int(row["x_t1"]), int(row["y_t1"]))
        cv2.arrowedLine(image, p0, p1, (0, 255, 0), 1, tipLength=0.35)
    for _, row in fg.iterrows():
        cv2.circle(image, (int(row["x_t"]), int(row["y_t"])), 3, (255, 0, 0), -1)
    summary = background_motion_summary(track_df)
    cv2.putText(image, f"bg dx={summary['bg_dx']:.2f}, dy={summary['bg_dy']:.2f}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)
    return image
def show_background_video(frames_batch, name, fps=8, clip_frames=48, seed=0):
    indices = clip_indices(max(0, len(frames_batch) - 1), clip_frames=clip_frames, seed=seed)
    rendered = []
    for i in indices:
        track_df = track_background_points(frames_batch[i], frames_batch[i + 1])
        rendered.append(render_background_tracking_frame(frames_batch[i], track_df))
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 2 if len(indices) else 0}")
    return show_video(np.asarray(rendered), name, fps=fps)
def show_background_experiment(frames_batch, frame_idx, title):
    next_idx = min(frame_idx + 1, len(frames_batch) - 1)
    track_df = track_background_points(frames_batch[frame_idx], frames_batch[next_idx])
    show_background_video(frames_batch, title.lower().replace(" ", "_").replace("-", "") + "_background.mp4", fps=8, clip_frames=48, seed=0)
    plot_background_motion(track_df, title)
    data = show_background_data(track_df, f"{title} data representation")
    return track_df, data


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_1_lunar_tracks, bg_1_lunar_data = show_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Dynamic Background Tracking",
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_2_car_tracks, bg_2_car_data = show_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Dynamic Background Tracking",
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_3_traffic_tracks, bg_3_traffic_data = show_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Dynamic Background Tracking",
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_4_youtube_tracks, bg_4_youtube_data = show_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Dynamic Background Tracking",
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_5_hard_tracks, bg_5_hard_data = show_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Dynamic Background Tracking",
)


## Camera Motion Compensation + Residual Motion


In [ ]:
# Camera-motion background segmentation
# Background = pixels explained by global camera motion; active agents = residual motion.
def estimate_camera_residual_background(frame_t, frame_t1, max_corners=800, residual_percentile=82):
    gray_t = gray_frame(frame_t)
    gray_t1 = gray_frame(frame_t1)
    points_t = cv2.goodFeaturesToTrack(gray_t, maxCorners=max_corners, qualityLevel=0.01, minDistance=6, blockSize=7)
    if points_t is None:
        bg_mask = np.ones(gray_t.shape, dtype=bool)
        return bg_mask, np.zeros_like(gray_t), np.eye(2, 3, dtype=np.float32)
    points_t1, status, _ = cv2.calcOpticalFlowPyrLK(gray_t, gray_t1, points_t, None)
    status = status.reshape(-1).astype(bool)
    src = points_t.reshape(-1, 2)[status]
    dst = points_t1.reshape(-1, 2)[status]
    if len(src) < 8:
        bg_mask = np.ones(gray_t.shape, dtype=bool)
        return bg_mask, np.zeros_like(gray_t), np.eye(2, 3, dtype=np.float32)
    affine, _ = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    if affine is None:
        affine = np.eye(2, 3, dtype=np.float32)
    warped = cv2.warpAffine(frame_t, affine, (frame_t.shape[1], frame_t.shape[0]))
    residual = cv2.cvtColor(cv2.absdiff(warped, frame_t1), cv2.COLOR_RGB2GRAY)
    threshold = max(8, np.percentile(residual, residual_percentile))
    fg_mask = residual > threshold
    kernel = np.ones((5, 5), np.uint8)
    fg_mask = cv2.morphologyEx(fg_mask.astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
    fg_mask = cv2.morphologyEx(fg_mask.astype(np.uint8), cv2.MORPH_DILATE, kernel).astype(bool)
    bg_mask = ~fg_mask
    return bg_mask, residual, affine
def render_background_mask(frame, bg_mask, title):
    overlay = frame.copy()
    overlay[bg_mask] = (overlay[bg_mask] * 0.55 + np.array([40, 180, 90]) * 0.45).astype(np.uint8)
    overlay[~bg_mask] = (overlay[~bg_mask] * 0.65 + np.array([230, 50, 40]) * 0.35).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(overlay)
    plt.title(title)
    plt.axis("off")
    plt.show()
def show_camera_residual_background_experiment(frames_batch, frame_idx, title):
    next_idx = min(frame_idx + 1, len(frames_batch) - 1)
    bg_mask, residual, affine = estimate_camera_residual_background(frames_batch[frame_idx], frames_batch[next_idx])
    render_background_mask(frames_batch[frame_idx], bg_mask, title)
    data = pd.DataFrame([{
        "frame": int(frame_idx),
        "background_pixels": int(bg_mask.sum()),
        "active_agent_pixels": int((~bg_mask).sum()),
        "background_ratio": float(bg_mask.mean()),
        "affine": affine.round(4).tolist(),
        "median_residual": float(np.median(residual)),
    }])
    display(data)
    return bg_mask, data


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_camera_residual_1_lunar_mask, bg_camera_residual_1_lunar_data = show_camera_residual_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Camera Motion Compensation + Residual Motion"
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_camera_residual_2_car_mask, bg_camera_residual_2_car_data = show_camera_residual_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Camera Motion Compensation + Residual Motion"
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_camera_residual_3_traffic_mask, bg_camera_residual_3_traffic_data = show_camera_residual_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Camera Motion Compensation + Residual Motion"
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_camera_residual_4_youtube_mask, bg_camera_residual_4_youtube_data = show_camera_residual_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Camera Motion Compensation + Residual Motion"
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_camera_residual_5_hard_mask, bg_camera_residual_5_hard_data = show_camera_residual_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Camera Motion Compensation + Residual Motion"
)


## OpenCV MOG2 Background Subtractor


In [ ]:
# OpenCV background-subtractor experiments
# Background = pixels classified as stable background by the subtractor.
def estimate_subtractor_background(frames_batch, frame_idx, method="MOG2", history=80):
    if method == "MOG2":
        subtractor = cv2.createBackgroundSubtractorMOG2(history=history, varThreshold=24, detectShadows=True)
    elif method == "KNN":
        subtractor = cv2.createBackgroundSubtractorKNN(history=history, dist2Threshold=500, detectShadows=True)
    else:
        raise ValueError(method)
    start = max(0, int(frame_idx) - history)
    fg_mask = None
    for frame in frames_batch[start:int(frame_idx) + 1]:
        fg_mask = subtractor.apply(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    if fg_mask is None:
        fg_mask = np.zeros(frames_batch[frame_idx].shape[:2], dtype=np.uint8)
    active_mask = fg_mask == 255
    kernel = np.ones((5, 5), np.uint8)
    active_mask = cv2.morphologyEx(active_mask.astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
    active_mask = cv2.morphologyEx(active_mask.astype(np.uint8), cv2.MORPH_DILATE, kernel).astype(bool)
    bg_mask = ~active_mask
    return bg_mask, fg_mask
def show_subtractor_background_experiment(frames_batch, frame_idx, title, method):
    bg_mask, fg_mask = estimate_subtractor_background(frames_batch, frame_idx, method=method)
    render_background_mask(frames_batch[frame_idx], bg_mask, title)
    data = pd.DataFrame([{
        "frame": int(frame_idx),
        "method": method,
        "background_pixels": int(bg_mask.sum()),
        "active_agent_pixels": int((~bg_mask).sum()),
        "background_ratio": float(bg_mask.mean()),
    }])
    display(data)
    return bg_mask, data


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_mog2_1_lunar_mask, bg_mog2_1_lunar_data = show_subtractor_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - OpenCV MOG2 Background Subtractor",
    method="MOG2"
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_mog2_2_car_mask, bg_mog2_2_car_data = show_subtractor_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - OpenCV MOG2 Background Subtractor",
    method="MOG2"
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_mog2_3_traffic_mask, bg_mog2_3_traffic_data = show_subtractor_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - OpenCV MOG2 Background Subtractor",
    method="MOG2"
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_mog2_4_youtube_mask, bg_mog2_4_youtube_data = show_subtractor_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - OpenCV MOG2 Background Subtractor",
    method="MOG2"
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_mog2_5_hard_mask, bg_mog2_5_hard_data = show_subtractor_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - OpenCV MOG2 Background Subtractor",
    method="MOG2"
)


## OpenCV KNN Background Subtractor


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_knn_1_lunar_mask, bg_knn_1_lunar_data = show_subtractor_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - OpenCV KNN Background Subtractor",
    method="KNN"
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_knn_2_car_mask, bg_knn_2_car_data = show_subtractor_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - OpenCV KNN Background Subtractor",
    method="KNN"
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_knn_3_traffic_mask, bg_knn_3_traffic_data = show_subtractor_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - OpenCV KNN Background Subtractor",
    method="KNN"
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_knn_4_youtube_mask, bg_knn_4_youtube_data = show_subtractor_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - OpenCV KNN Background Subtractor",
    method="KNN"
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_knn_5_hard_mask, bg_knn_5_hard_data = show_subtractor_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - OpenCV KNN Background Subtractor",
    method="KNN"
)


## SAM2 Video Segmentation


In [ ]:
# Video segmentation model adapter
# Use this for SAM2/XMem once the model checkpoint is wired in.
def show_video_segmentation_background_experiment(frames_batch, frame_idx, title, model_name, active_agent_mask_fn=None):
    if active_agent_mask_fn is None:
        print(f"{model_name}: active_agent_mask_fn is not configured yet")
        print("Expected output: active_agent_mask where agents=True; background is the inverse mask.")
        bg_mask = np.ones(frames_batch[frame_idx].shape[:2], dtype=bool)
    else:
        active_mask = active_agent_mask_fn(frames_batch, frame_idx).astype(bool)
        bg_mask = ~active_mask
    render_background_mask(frames_batch[frame_idx], bg_mask, title)
    data = pd.DataFrame([{
        "frame": int(frame_idx),
        "model": model_name,
        "background_pixels": int(bg_mask.sum()),
        "active_agent_pixels": int((~bg_mask).sum()),
        "background_ratio": float(bg_mask.mean()),
        "configured": active_agent_mask_fn is not None,
    }])
    display(data)
    return bg_mask, data


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_sam2_1_lunar_mask, bg_sam2_1_lunar_data = show_video_segmentation_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - SAM2 Video Segmentation",
    model_name="SAM2"
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_sam2_2_car_mask, bg_sam2_2_car_data = show_video_segmentation_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - SAM2 Video Segmentation",
    model_name="SAM2"
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_sam2_3_traffic_mask, bg_sam2_3_traffic_data = show_video_segmentation_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - SAM2 Video Segmentation",
    model_name="SAM2"
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_sam2_4_youtube_mask, bg_sam2_4_youtube_data = show_video_segmentation_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - SAM2 Video Segmentation",
    model_name="SAM2"
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_sam2_5_hard_mask, bg_sam2_5_hard_data = show_video_segmentation_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - SAM2 Video Segmentation",
    model_name="SAM2"
)


## XMem Video Object Segmentation


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_xmem_1_lunar_mask, bg_xmem_1_lunar_data = show_video_segmentation_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - XMem Video Object Segmentation",
    model_name="XMem"
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_xmem_2_car_mask, bg_xmem_2_car_data = show_video_segmentation_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - XMem Video Object Segmentation",
    model_name="XMem"
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_xmem_3_traffic_mask, bg_xmem_3_traffic_data = show_video_segmentation_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - XMem Video Object Segmentation",
    model_name="XMem"
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_xmem_4_youtube_mask, bg_xmem_4_youtube_data = show_video_segmentation_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - XMem Video Object Segmentation",
    model_name="XMem"
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_xmem_5_hard_mask, bg_xmem_5_hard_data = show_video_segmentation_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - XMem Video Object Segmentation",
    model_name="XMem"
)


## OpenAI VLM Agent Segmentation -> Background Mask


In [ ]:
# OpenAI VLM agent segmentation -> background mask
# Active agents are segmented first; background is the inverse mask.
import base64
import io
import os
import re
from dotenv import load_dotenv
load_dotenv(".env")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
VLM_BACKGROUND_MODEL = os.getenv("OPENAI_VLM_MODEL", "gpt-4.1-mini")
def frame_to_data_url(frame, max_width=960, quality=85):
    image = Image.fromarray(frame.astype(np.uint8))
    original_width, original_height = image.size
    scale = min(1.0, max_width / float(original_width))
    if scale < 1.0:
        image = image.resize((int(original_width * scale), int(original_height * scale)))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality)
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}", scale
def parse_vlm_json(text):
    text = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL)
    if fenced:
        text = fenced.group(1)
    return json.loads(text)
def call_vlm_active_agent_polygons(frame, model=VLM_BACKGROUND_MODEL):
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is not set, so the VLM segmentation test cannot run.")
    from openai import OpenAI
    data_url, scale = frame_to_data_url(frame)
    height, width = frame.shape[:2]
    prompt = f"""
You are segmenting one driving-scene frame for background extraction.
Return only strict JSON, no markdown.
Goal: identify active agents only. Active agents include vehicles, buses, taxis, cyclists, pedestrians, animals, and other independently moving objects.
Background includes road, lane markings, sidewalks, buildings, sky, traffic lights, signs, poles, trees, median, curbs, parked-looking static infrastructure.
Use pixel coordinates in the original image size: width={width}, height={height}.
Return approximate polygons around every visible active agent.
JSON schema:
{{"active_agents":[{{"label":"car|bus|truck|person|bike|other", "confidence":0.0, "polygon":[[x,y],[x,y],[x,y]]}}]}}
""".strip()
    client = OpenAI()
    response = client.responses.create(
        model=model,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": data_url},
            ],
        }],
    )
    data = parse_vlm_json(response.output_text)
    for agent in data.get("active_agents", []):
        agent["polygon"] = [[float(x), float(y)] for x, y in agent.get("polygon", [])]
    return data
def agents_to_mask(frame, agents, min_confidence=0.2):
    height, width = frame.shape[:2]
    mask_image = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask_image)
    for agent in agents:
        if float(agent.get("confidence", 1.0)) < min_confidence:
            continue
        polygon = agent.get("polygon", [])
        if len(polygon) >= 3:
            draw.polygon([(int(x), int(y)) for x, y in polygon], fill=255)
    return np.asarray(mask_image) > 0
def show_vlm_background_experiment(frames_batch, frame_idx, title, model=VLM_BACKGROUND_MODEL):
    frame = frames_batch[frame_idx]
    skipped = False
    error = None
    try:
        vlm_data = call_vlm_active_agent_polygons(frame, model=model)
        active_mask = agents_to_mask(frame, vlm_data.get("active_agents", []))
    except Exception as exc:
        skipped = True
        error = f"{type(exc).__name__}: {exc}"
        print(f"Skipping OpenAI VLM test: {error}")
        vlm_data = {"active_agents": []}
        active_mask = np.zeros(frame.shape[:2], dtype=bool)
    bg_mask = ~active_mask
    render_background_mask(frame, bg_mask, title)
    rows = []
    for i, agent in enumerate(vlm_data.get("active_agents", [])):
        rows.append({
            "frame": int(frame_idx),
            "agent_id": i,
            "label": agent.get("label", "other"),
            "confidence": float(agent.get("confidence", 1.0)),
            "polygon_points": len(agent.get("polygon", [])),
        })
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "model": model,
        "skipped": skipped,
        "error": error,
        "agents": len(vlm_data.get("active_agents", [])),
        "background_pixels": int(bg_mask.sum()),
        "active_agent_pixels": int(active_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
    }])
    display(summary)
    display(pd.DataFrame(rows))
    return bg_mask, summary


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_vlm_1_lunar_mask, bg_vlm_1_lunar_data = show_vlm_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - OpenAI VLM Agent Segmentation -> Background Mask"
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_vlm_2_car_mask, bg_vlm_2_car_data = show_vlm_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - OpenAI VLM Agent Segmentation -> Background Mask"
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_vlm_3_traffic_mask, bg_vlm_3_traffic_data = show_vlm_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - OpenAI VLM Agent Segmentation -> Background Mask"
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_vlm_4_youtube_mask, bg_vlm_4_youtube_data = show_vlm_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - OpenAI VLM Agent Segmentation -> Background Mask"
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_vlm_5_hard_mask, bg_vlm_5_hard_data = show_vlm_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - OpenAI VLM Agent Segmentation -> Background Mask"
)


## Free/Open-Source VLM Agent Segmentation -> Background Mask


In [ ]:
# Free/open-source VLM agent segmentation adapter
# Candidate models: Qwen2.5-VL, InternVL, LLaVA-OneVision. They can describe active agents,
# but exact masks still need polygons from the model or a paired segmenter.
def show_free_vlm_background_experiment(frames_batch, frame_idx, title, active_agent_polygons=None, model_name="Qwen2.5-VL / local open-source VLM"):
    frame = frames_batch[frame_idx]
    if active_agent_polygons is None:
        print(f"{model_name}: local VLM is not configured yet")
        print("Provide active_agent_polygons=[{'label': ..., 'confidence': ..., 'polygon': [[x,y], ...]}] or wire a local model call here.")
        active_agent_polygons = []
    active_mask = agents_to_mask(frame, active_agent_polygons)
    bg_mask = ~active_mask
    render_background_mask(frame, bg_mask, title)
    data = pd.DataFrame([{
        "frame": int(frame_idx),
        "model": model_name,
        "configured": bool(active_agent_polygons),
        "agents": len(active_agent_polygons),
        "background_pixels": int(bg_mask.sum()),
        "active_agent_pixels": int(active_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
    }])
    display(data)
    return bg_mask, data


### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_free_vlm_1_lunar_mask, bg_free_vlm_1_lunar_data = show_free_vlm_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Free/Open-Source VLM Agent Segmentation -> Background Mask"
)


### 2 - Car Racing Background Segmentation


In [ ]:
bg_free_vlm_2_car_mask, bg_free_vlm_2_car_data = show_free_vlm_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Free/Open-Source VLM Agent Segmentation -> Background Mask"
)


### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_free_vlm_3_traffic_mask, bg_free_vlm_3_traffic_data = show_free_vlm_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Free/Open-Source VLM Agent Segmentation -> Background Mask"
)


### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_free_vlm_4_youtube_mask, bg_free_vlm_4_youtube_data = show_free_vlm_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Free/Open-Source VLM Agent Segmentation -> Background Mask"
)


### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_free_vlm_5_hard_mask, bg_free_vlm_5_hard_data = show_free_vlm_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Free/Open-Source VLM Agent Segmentation -> Background Mask"
)


## Free API VLM Model Comparison -> Background Mask


In [ ]:
# Free API VLM model comparison via OpenRouter
# These models are tested through API now, but have downloadable/open weights for later local runs.
import base64
import io
import os
import re
import requests
from dotenv import load_dotenv
load_dotenv(".env")
print("OPENROUTER_API_KEY loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
FREE_API_VLM_MODELS = {
    "qwen2_5_vl_72b": "qwen/qwen2.5-vl-72b-instruct:free",
    "llama_3_2_vision_11b": "meta-llama/llama-3.2-11b-vision-instruct:free",
    "gemma_4_31b": "google/gemma-4-31b-it:free",
    "nemotron_nano_12b_vl": "nvidia/nemotron-nano-12b-v2-vl:free",
}
FREE_API_VLM_LOCAL_WEIGHTS = {
    "qwen2_5_vl_72b": "Qwen/Qwen2.5-VL-72B-Instruct",
    "llama_3_2_vision_11b": "meta-llama/Llama-3.2-11B-Vision-Instruct",
    "gemma_4_31b": "google/gemma-4-31B-it",
    "nemotron_nano_12b_vl": "nvidia/NVIDIA-Nemotron-Nano-12B-v2-VL-BF16",
}
def frame_to_openrouter_image_url(frame, max_width=960, quality=85):
    image = Image.fromarray(frame.astype(np.uint8))
    original_width, original_height = image.size
    scale = min(1.0, max_width / float(original_width))
    if scale < 1.0:
        image = image.resize((int(original_width * scale), int(original_height * scale)))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality)
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"
def parse_free_api_vlm_json(text):
    text = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()
    return json.loads(text)
def call_free_api_vlm_agents(frame, model_key, timeout=120):
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set")
    model = FREE_API_VLM_MODELS[model_key]
    height, width = frame.shape[:2]
    prompt = f"""
You are segmenting one driving-scene frame for background extraction.
Return only strict JSON, no markdown.
Goal: identify active agents only. Active agents include vehicles, buses, taxis, cyclists, pedestrians, animals, and other independently moving objects.
Background includes road, lane markings, sidewalks, buildings, sky, traffic lights, signs, poles, trees, median, curbs, parked-looking static infrastructure.
Use original image pixel coordinates: width={width}, height={height}.
Return approximate polygons around every visible active agent.
JSON schema:
{{"active_agents":[{{"label":"car|bus|truck|person|bike|other", "confidence":0.0, "polygon":[[x,y],[x,y],[x,y]]}}]}}
""".strip()
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "http://localhost/action-inference",
            "X-Title": "Action Inference Free VLM Background Test",
        },
        json={
            "model": model,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": frame_to_openrouter_image_url(frame)}},
                ],
            }],
            "temperature": 0,
            "max_tokens": 1600,
        },
        timeout=timeout,
    )
    response.raise_for_status()
    text = response.json()["choices"][0]["message"]["content"]
    return parse_free_api_vlm_json(text)
def show_free_api_vlm_background_experiment(frames_batch, frame_idx, title, model_key):
    frame = frames_batch[frame_idx]
    skipped = False
    error = None
    try:
        vlm_data = call_free_api_vlm_agents(frame, model_key=model_key)
        agents = vlm_data.get("active_agents", [])
        active_mask = agents_to_mask(frame, agents)
    except Exception as exc:
        skipped = True
        error = f"{type(exc).__name__}: {exc}"
        print(f"Skipping {FREE_API_VLM_MODELS[model_key]}: {error}")
        agents = []
        active_mask = np.zeros(frame.shape[:2], dtype=bool)
    bg_mask = ~active_mask
    render_background_mask(frame, bg_mask, title)
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "api_model": FREE_API_VLM_MODELS[model_key],
        "local_weights": FREE_API_VLM_LOCAL_WEIGHTS[model_key],
        "skipped": skipped,
        "error": error,
        "agents": len(agents),
        "background_pixels": int(bg_mask.sum()),
        "active_agent_pixels": int(active_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
    }])
    display(summary)
    display(pd.DataFrame(agents))
    return bg_mask, summary


In [ ]:
# Minimal frame loader for running only this Free API VLM section after a kernel restart.
# It creates frames/frame_idx variables only when they are missing.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw

try:
    import cv2
except Exception as exc:
    cv2 = None
    print(f"cv2 unavailable: {type(exc).__name__}: {exc}")

def load_video_frames_minimal(path, max_frames=120, stride=3, start_seconds=0):
    path = Path(path)
    if cv2 is not None:
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_seconds * fps))
        out = []
        frame_no = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_no % stride == 0:
                out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if len(out) >= max_frames:
                    break
            frame_no += 1
        cap.release()
        if out:
            return np.stack(out)
    reader = imageio.get_reader(path)
    out = []
    for frame_no, frame in enumerate(reader):
        if frame_no % stride == 0:
            out.append(np.asarray(frame)[..., :3])
            if len(out) >= max_frames:
                break
    reader.close()
    if not out:
        raise RuntimeError(f"No frames loaded from {path}")
    return np.stack(out)

def ensure_frames_var(var_name, path, idx_name, idx_value, max_frames=120, stride=3, start_seconds=0):
    if var_name not in globals():
        globals()[var_name] = load_video_frames_minimal(path, max_frames=max_frames, stride=stride, start_seconds=start_seconds)
        print(f"loaded {var_name}:", globals()[var_name].shape)
    if idx_name not in globals():
        globals()[idx_name] = min(idx_value, len(globals()[var_name]) - 1)
        print(f"set {idx_name}:", globals()[idx_name])

def render_background_mask(frame, bg_mask, title):
    overlay = frame.copy()
    overlay[bg_mask] = (overlay[bg_mask] * 0.55 + np.array([40, 180, 90]) * 0.45).astype(np.uint8)
    overlay[~bg_mask] = (overlay[~bg_mask] * 0.65 + np.array([230, 50, 40]) * 0.35).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(overlay)
    plt.title(title)
    plt.axis("off")
    plt.show()

def agents_to_mask(frame, agents, min_confidence=0.2):
    height, width = frame.shape[:2]
    mask_image = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask_image)
    for agent in agents:
        if float(agent.get("confidence", 1.0)) < min_confidence:
            continue
        polygon = agent.get("polygon", [])
        if len(polygon) >= 3:
            draw.polygon([(int(x), int(y)) for x, y in polygon], fill=255)
    return np.asarray(mask_image) > 0

# These are enough to run the Free API VLM cells without running the whole notebook.
ensure_frames_var("frames", "generated_videos/1_lunar_lander.gif", "frame_idx", 10, max_frames=80, stride=1)
ensure_frames_var("frames_car", "generated_videos/2_car_racing.gif", "car_frame_idx", 40, max_frames=120, stride=1)
ensure_frames_var("frames_medium", "external_videos/traffic.avi", "traffic_frame_idx", 1, max_frames=120, stride=1)
ensure_frames_var("frames_problem4", "external_videos/problem_4_youtube_random.mp4", "problem4_frame_idx", 30, max_frames=120, stride=3, start_seconds=20)
ensure_frames_var("frames_problem5", "external_videos/problem_5_youtube_hardest.mp4", "problem5_frame_idx", 40, max_frames=120, stride=3, start_seconds=20)



### Qwen2.5-VL 72B


#### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_free_api_qwen2_5_vl_72b_1_lunar_mask, bg_free_api_qwen2_5_vl_72b_1_lunar_data = show_free_api_vlm_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Qwen2.5-VL 72B Free API VLM Background Mask",
    model_key="qwen2_5_vl_72b"
)


#### 2 - Car Racing Background Segmentation


In [ ]:
bg_free_api_qwen2_5_vl_72b_2_car_mask, bg_free_api_qwen2_5_vl_72b_2_car_data = show_free_api_vlm_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Qwen2.5-VL 72B Free API VLM Background Mask",
    model_key="qwen2_5_vl_72b"
)


#### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_free_api_qwen2_5_vl_72b_3_traffic_mask, bg_free_api_qwen2_5_vl_72b_3_traffic_data = show_free_api_vlm_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Qwen2.5-VL 72B Free API VLM Background Mask",
    model_key="qwen2_5_vl_72b"
)


#### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_free_api_qwen2_5_vl_72b_4_youtube_mask, bg_free_api_qwen2_5_vl_72b_4_youtube_data = show_free_api_vlm_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Qwen2.5-VL 72B Free API VLM Background Mask",
    model_key="qwen2_5_vl_72b"
)


#### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_free_api_qwen2_5_vl_72b_5_hard_mask, bg_free_api_qwen2_5_vl_72b_5_hard_data = show_free_api_vlm_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Qwen2.5-VL 72B Free API VLM Background Mask",
    model_key="qwen2_5_vl_72b"
)


### Llama 3.2 Vision 11B


#### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_free_api_llama_3_2_vision_11b_1_lunar_mask, bg_free_api_llama_3_2_vision_11b_1_lunar_data = show_free_api_vlm_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Llama 3.2 Vision 11B Free API VLM Background Mask",
    model_key="llama_3_2_vision_11b"
)


#### 2 - Car Racing Background Segmentation


In [ ]:
bg_free_api_llama_3_2_vision_11b_2_car_mask, bg_free_api_llama_3_2_vision_11b_2_car_data = show_free_api_vlm_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Llama 3.2 Vision 11B Free API VLM Background Mask",
    model_key="llama_3_2_vision_11b"
)


#### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_free_api_llama_3_2_vision_11b_3_traffic_mask, bg_free_api_llama_3_2_vision_11b_3_traffic_data = show_free_api_vlm_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Llama 3.2 Vision 11B Free API VLM Background Mask",
    model_key="llama_3_2_vision_11b"
)


#### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_free_api_llama_3_2_vision_11b_4_youtube_mask, bg_free_api_llama_3_2_vision_11b_4_youtube_data = show_free_api_vlm_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Llama 3.2 Vision 11B Free API VLM Background Mask",
    model_key="llama_3_2_vision_11b"
)


#### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_free_api_llama_3_2_vision_11b_5_hard_mask, bg_free_api_llama_3_2_vision_11b_5_hard_data = show_free_api_vlm_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Llama 3.2 Vision 11B Free API VLM Background Mask",
    model_key="llama_3_2_vision_11b"
)


### Gemma 4 31B Vision


#### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_free_api_gemma_4_31b_1_lunar_mask, bg_free_api_gemma_4_31b_1_lunar_data = show_free_api_vlm_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Gemma 4 31B Vision Free API VLM Background Mask",
    model_key="gemma_4_31b"
)


#### 2 - Car Racing Background Segmentation


In [ ]:
bg_free_api_gemma_4_31b_2_car_mask, bg_free_api_gemma_4_31b_2_car_data = show_free_api_vlm_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Gemma 4 31B Vision Free API VLM Background Mask",
    model_key="gemma_4_31b"
)


#### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_free_api_gemma_4_31b_3_traffic_mask, bg_free_api_gemma_4_31b_3_traffic_data = show_free_api_vlm_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Gemma 4 31B Vision Free API VLM Background Mask",
    model_key="gemma_4_31b"
)


#### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_free_api_gemma_4_31b_4_youtube_mask, bg_free_api_gemma_4_31b_4_youtube_data = show_free_api_vlm_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Gemma 4 31B Vision Free API VLM Background Mask",
    model_key="gemma_4_31b"
)


#### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_free_api_gemma_4_31b_5_hard_mask, bg_free_api_gemma_4_31b_5_hard_data = show_free_api_vlm_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Gemma 4 31B Vision Free API VLM Background Mask",
    model_key="gemma_4_31b"
)


### Nemotron Nano 12B VL


#### 1 - Lunar Lander Background Segmentation


In [ ]:
bg_free_api_nemotron_nano_12b_vl_1_lunar_mask, bg_free_api_nemotron_nano_12b_vl_1_lunar_data = show_free_api_vlm_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Nemotron Nano 12B VL Free API VLM Background Mask",
    model_key="nemotron_nano_12b_vl"
)


#### 2 - Car Racing Background Segmentation


In [ ]:
bg_free_api_nemotron_nano_12b_vl_2_car_mask, bg_free_api_nemotron_nano_12b_vl_2_car_data = show_free_api_vlm_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Nemotron Nano 12B VL Free API VLM Background Mask",
    model_key="nemotron_nano_12b_vl"
)


#### 3 - Recorded Traffic With People Background Segmentation


In [ ]:
bg_free_api_nemotron_nano_12b_vl_3_traffic_mask, bg_free_api_nemotron_nano_12b_vl_3_traffic_data = show_free_api_vlm_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Nemotron Nano 12B VL Free API VLM Background Mask",
    model_key="nemotron_nano_12b_vl"
)


#### 4 - Random YouTube Driving Scene Background Segmentation


In [ ]:
bg_free_api_nemotron_nano_12b_vl_4_youtube_mask, bg_free_api_nemotron_nano_12b_vl_4_youtube_data = show_free_api_vlm_background_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Nemotron Nano 12B VL Free API VLM Background Mask",
    model_key="nemotron_nano_12b_vl"
)


#### 5 - Hard Vehicle-Crowd Interaction Background Segmentation


In [ ]:
bg_free_api_nemotron_nano_12b_vl_5_hard_mask, bg_free_api_nemotron_nano_12b_vl_5_hard_data = show_free_api_vlm_background_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Nemotron Nano 12B VL Free API VLM Background Mask",
    model_key="nemotron_nano_12b_vl"
)
